# Lesson 26 · From language model to assistant: SFT, reinforcement learning with a checker, reward hacking and DPO

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/26-hau-huan-luyen-en.ipynb)

This notebook accompanies [Lesson 26 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/hau-huan-luyen/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/26-hau-huan-luyen.ipynb)):
the code, the data and every check are the same. The reel and the interactive demo are on the lesson page.
After pretraining, the model only knows how to continue text. Post-training teaches it to answer, and teaches it with
feedback, not only with examples. We start with the real GPT-2 small (precomputed results), then post-train the
number-reading model of Lesson 16. In that task a program can grade every answer, so we measure exactly the things
that can only be measured approximately for a real chatbot: the reward, the KL divergence, reward hacking. The model
reads how a number is read aloud in Vietnamese and writes its digits; those readings are its data, so they stay
Vietnamese here, and section 4 glosses them.

**What you'll do:**
1. See how GPT-2 small answers (or doesn't answer) a question, and why two question–answer examples change the
   result completely.
2. Write two graders for the number-reading task, one exact and one sloppy, then run the demo's three post-trained
   models.
3. Write GRPO-style reinforcement learning yourself, simplified (REINFORCE with a within-group baseline and a KL
   penalty), check its gradient, then run 100 real steps.
4. See reward hacking (Goodhart's law), compute the closed-form solution $\pi^* \propto \pi_{\text{ref}}\,e^{r/\beta}$,
   and learn DPO on a small table.
5. Measure test-time compute: majority voting, best-of-N and its exact KL divergence.

**Background:** [Lesson 22 · Pretraining & fine-tuning](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/tien-huan-luyen/),
[Lesson 16 · The attention mechanism](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/attention/) (the
number-reading model we post-train) and the KL divergence of
[Lesson 19](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/autoencoder-vae/).
**Run time:** under a minute on a CPU, including the first download of the four results files (~0.6 MB).

Every number in the notebook is computed by the cells below (or read from the results files the demo uses), and
matches the numbers in the demo.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `fmt()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign.

In [ ]:
import base64
import hashlib
import json
import math
import urllib.request
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def fmt(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38 (never prints “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}"
    return s.replace("-", "−")


def pct(p, digits=2):
    return fmt(100 * p, digits) + "%"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return fmt(v, len(s.split(".")[1]) if "." in s else 0)


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def log_tick(v, _pos=None):
    """A log-axis tick label: 0.5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Put the tick labels of every axis of the figure (colorbars included) in the English format; call right before
    plt.show(). A log axis gets 1–2–5 ticks if it is short, or powers of 10 (evenly spaced, at most ~9 ticks) if it is
    long."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def prob(v):
    """A probability: a percentage from 0.01% up, written like 4.5·10⁻⁵ below that."""
    return pct(v, 2) if v >= 1e-4 else sci(v)


def sci(v):
    """A small positive number like 3.1·10⁻⁹."""
    e = math.floor(math.log10(v))
    return f"{fmt(v / 10**e, 1)}·10{str(e).translate(SUP)}"


def span(vals, f=pct):
    """The lowest–highest range across seeds."""
    lo, hi = f(min(vals)), f(max(vals))
    return lo if lo == hi else f"{lo}–{hi}"


print(fmt(53259, 0), fmt(0.0047, 4), pct(0.6735), prob(4.5e-5))

## 2 · The data: the demo's four results files

GPT-2 small (124 million parameters) is too large for this notebook, and training the number-reading models takes
tens of minutes. The cell below downloads exactly the four files the demo uses and checks their MD5 hashes:
- `posttrain-base.json`: what the real GPT-2 small writes after 9 prompts (computed with the real model of
  Lesson 17);
- `posttrain-weights.json`: the float16 weights of the three number-reading models (seed 0): after SFT, after
  reinforcement learning with the exact checker, and after reinforcement learning with the sloppy grader;
- `posttrain-rl.json`: the test results of every run (3 seeds per configuration), used in sections 5–7 and 10–12;
- `posttrain-cands.json`: the 12 questions of the demo's second card, with the common answers and their
  probabilities.

In [ ]:
# Results of the long runs (the same files as the demo on the web), downloaded from the data/ folder of the public
# notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


base = fetch_json("posttrain-base.json", "fede87d4b05037589e51fe2d63c57351")  # the real GPT-2 small
shipped = fetch_json("posttrain-weights.json", "65903d4b01563135629bfa515122ab57")  # the demo's three models
runs = fetch_json("posttrain-rl.json", "abfd281bb58763ae7f139a316f10fe7d")  # every SFT / reinforcement-learning run
cands = fetch_json("posttrain-cands.json", "d484589eac25dff1125ea1005706c827")  # the 12 questions of card 2
names = ("posttrain-base.json", "posttrain-weights.json", "posttrain-rl.json", "posttrain-cands.json")
size = sum((Path("data") / n).stat().st_size for n in names)
print(f"{len(names)} files, {fmt(size / 1e6, 1)} MB · models:", list(shipped["models"]))

## 3 · The base model continues, it doesn't answer

GPT-2 small was only pretrained to guess the next token (Lesson 17). The file `posttrain-base.json` records what it
actually does with 9 prompts, each placed right after the token `<|endoftext|>`:
- the probability of the first token right after the prompt, and the 5 most probable tokens;
- 41 continuations of 40 tokens each: 1 greedy, 20 with temperature $T = 0.7$ and 20 with $T = 1$, top-k 40 (the
  sampling of Lesson 17, 20 seeds).

The example question is “What is the capital of France?”. We care about the token “ Paris” (with a leading space,
exactly one token in GPT-2's vocabulary of 50,257 tokens).

### 🤔 Predict first
Right after the bare question “What is the capital of France?”, how much probability does GPT-2 small put on the
token “ Paris”: over 50%, about one in ten, or under 1%? And when the same question comes after two question–answer
examples (Italy → Rome, Japan → Tokyo)?

In [ ]:
P = {p["id"]: p for p in base["prompts"]}
LABEL = {
    "q_en": "bare question",
    "qa_en": "Q: … A:",
    "chat_en": "User: … Assistant:",
    "few_en": "2 Q&A examples + question",
    "stmt_en": "“The capital of France is”",
    "q_vi": "question in Vietnamese",  # “Thủ đô của Pháp là gì?” (What is the capital of France?)
}
paris = {}
for pid, label in LABEL.items():
    p = P[pid]
    paris[pid] = sum("Paris" in c["segs"][0] for c in p["cont"])  # continuation before <|endoftext|> mentions Paris
    print(
        f"{label:>28}: p(“ Paris”) = {prob(p['p']):>9}, rank {p['rank']:>5}"
        f" · {paris[pid]}/{len(p['cont'])} continuations mention “Paris”"
    )
print(f"vocab: {fmt(base['setup']['vocab'], 0)} tokens")

For the bare question, “ Paris” has a probability of only 0.47%, in 17th place. Put the same question after two
question–answer examples and it rises to 67.35%, in first place. The model **does** know Paris; what it lacks is
*the habit of answering*. The cell below prints the 5 most probable tokens in three cases (the two ways of asking
above and the sentence start “The capital of France is”), and a few real continuations.

In [ ]:
def show(text, width=110):
    """One continuation on one line: ⏎ stands for a line break."""
    s = text.replace("\n", " ⏎ ").strip()
    return s if len(s) <= width else s[: width - 1] + "…"


for pid in ("q_en", "few_en", "stmt_en"):
    top = " · ".join(f"{t!r} {pct(v)}" for t, v in P[pid]["top5"])
    print(f"{LABEL[pid]}: {top}")
print()
for pid, setting, seed in (("q_en", "greedy", 0), ("q_en", "T0.7", 0), ("q_en", "T0.7", 1), ("few_en", "T0.7", 0)):
    c = next(c for c in P[pid]["cont"] if c["setting"] == setting and c["seed"] == seed)
    print(f"[{LABEL[pid]}, {setting}, seed {seed}]  {show(c['segs'][0])}")
new_q = sum(c["newQ"] for c in P["few_en"]["cont"])
print(f"\nafter 2 examples: {new_q}/{len(P['few_en']['cont'])} continuations write a new “Q:” line of their own")

For the bare question, the most probable token is a line break (49.03%): GPT-2 treats the question as the beginning
of a web page and continues that page. The greedy version repeats “The capital of France is Paris.” forever; one
sample at $T = 0.7$ says the capital is Lausanne (a Swiss city). After two question–answer examples, it answers
Paris but doesn't stop: it continues *the whole question–answer page*, with a new question it makes up itself.

The figure below plots the probability of “ Paris” (log scale) for the English prompts: blue bars are the prompts
without examples, the orange hatched bar is the prompt with two examples.

In [ ]:
order = ["stmt_en", "q_en", "chat_en", "qa_en", "few_en"]
vals = [P[k]["p"] for k in order]
fig, ax = plt.subplots(figsize=(8, 2.8))
bars = ax.barh(range(len(order)), vals, color=[ORANGE if k == "few_en" else BLUE for k in order])
bars[-1].set_hatch("//")
for i, v in enumerate(vals):
    ax.text(v * 1.15, i, prob(v), va="center", fontsize=9)
ax.set_yticks(range(len(order)), labels=[LABEL[k] for k in order])
ax.set(xscale="log", xlim=(1e-3, 3), xlabel="p(“ Paris”) right after the prompt (log scale)")
ax.set_title("GPT-2 small knows Paris, but only says it when the prompt looks like a Q&A page")
fmt_axes(fig)
plt.show()

Even the sentence “The capital of France is” gives “ Paris” only 0.16%: the model often writes “now”, “the”, “a”…,
because on the web that sentence often doesn't end there. We see two things:
- **The knowledge is already there from pretraining.** The two examples don't teach GPT-2 anything new (there is no
  gradient); they only select the *kind of text* the model continues.
- **Post-training teaches behavior.** The first step is supervised fine-tuning (SFT): still the cross-entropy of
  Lesson 22, but on example prompt–answer pairs, so that the model answers and then stops. InstructGPT used about
  13 thousand prompts with answers written by people; LIMA (Zhou et al., 2023) showed that 1,000 carefully chosen
  examples are enough to teach the format.

Measuring the quality of a real chatbot's answers is hard: it takes human raters, or a grading model. The rest of the
notebook uses a task where **a program can grade exactly**, so we can measure everything.

## 4 · The number-reading task and two graders

Lesson 16's model reads the Vietnamese reading of a number with 1–12 digits (“năm nghìn hai trăm”, five thousand two
hundred) and writes the digits (5200), ending with the symbol ⟨end⟩. The cell below copies Lesson 16's number reader
and sampler (identical to `reel/attention/numvi.py`): each digit is 0 with probability 0.25, each three-digit group
below the first one is 000 with probability 0.2.

To follow the readings: the digits 0–9 are “không, một, hai, ba, bốn, năm, sáu, bảy, tám, chín”; “trăm”, “nghìn”,
“triệu”, “tỷ” are hundred, thousand, million, billion; “mươi” marks the tens (“năm mươi”, fifty) and “mười” is ten;
after “mươi” the digits 1, 4, 5 are read “mốt”, “tư”, “lăm”, and after “mười” 5 is read “lăm”; “linh” marks an empty
tens place; and a three-digit group after the first keeps its hundreds even when they are 0 (“không trăm”, zero
hundred), while a 000 group is dropped. Lesson 16 explains these rules.

In [ ]:
D = ["không", "một", "hai", "ba", "bốn", "năm", "sáu", "bảy", "tám", "chín"]
SCALE = ["", "nghìn", "triệu", "tỷ"]


def tens_units(b, c):
    """The tens digit b and the units digit c of a three-digit group."""
    if b == 0:
        return ["linh", D[c]] if c else []
    if b == 1:
        return ["mười"] + (["lăm" if c == 5 else D[c]] if c else [])
    return [D[b], "mươi"] + ([{1: "mốt", 4: "tư", 5: "lăm"}.get(c, D[c])] if c else [])


def group(g, lead):
    a, b, c = g // 100, g // 10 % 10, g % 10
    if lead and a == 0:  # the first group doesn't read a leading 0
        return [D[c]] if b == 0 else tens_units(b, c)
    return [D[a], "trăm"] + tens_units(b, c)


def read(n):
    """Integer n ≥ 0 → list of words."""
    if n == 0:
        return ["không"]
    groups = []
    while n:
        groups.append(n % 1000)
        n //= 1000
    words = []
    for k in reversed(range(len(groups))):
        lead = k == len(groups) - 1
        if groups[k] == 0 and not lead:  # skip the 000 groups in the middle
            continue
        words += group(groups[k], lead)
        if SCALE[k]:
            words.append(SCALE[k])
    return words


def sample(rng, d, n, p0=0.25, pg=0.2):
    """n numbers with exactly d digits; each row holds the digits."""
    digs = np.where(rng.random((n, d)) < p0, 0, rng.integers(1, 10, (n, d)))
    digs[:, 0] = rng.integers(1, 10, n)  # the first digit is nonzero
    for g0 in range(d - 3, 0, -3):  # the three-digit groups counted from the right, except the first group
        digs[rng.random(n) < pg, g0 : g0 + 3] = 0
    return digs


for n in (5200, 150, 30500, 554000):
    print(f"{fmt(n, 0):>8} → {' '.join(read(n))}")
print(f"sampler: each digit is 0 with probability {fmt(0.25)}, each 000 group with probability {fmt(0.2, 1)}")
assert " ".join(read(7090400)) == "bảy triệu không trăm chín mươi nghìn bốn trăm"

Reinforcement learning needs a **reward** for each answer. Here there is an exact checker that **needs no answer
key**: it reads the digits of the answer aloud with `read()` and compares them with the words of the question. The
answer must end with ⟨end⟩ within 13 tokens, and must not start with 0 (except the number 0 itself).

The second is a **sloppy** grader, the kind of approximate measure often met in real life: it compares only the
nonzero digits with the words for nonzero digits in the question, in the same order, and ignores every digit 0. Both
are identical to `reel/posttrain/rl.py`.

Output tokens: 0–9 are the digits, 10 is ⟨end⟩; the decoder starts from symbol 11.

In [ ]:
EOS, BOS, TMAX, N_OUT = 10, 11, 13, 11
NZ = {"một": 1, "mốt": 1, "hai": 2, "ba": 3, "bốn": 4, "tư": 4, "năm": 5, "lăm": 5, "sáu": 6, "bảy": 7, "tám": 8}
NZ |= {"chín": 9, "mười": 1}  # each word names one nonzero digit (“mười”, ten, is the digit 1 in the tens place)


def tokens(digits):
    """A string of digits → the tokens of the correct answer, ending with ⟨end⟩."""
    return [int(c) for c in digits] + [EOS]


def answer_len(out):
    """The number of tokens up to the first ⟨end⟩ (13 if there is none)."""
    has = out == EOS
    return np.where(has.any(1), has.argmax(1) + 1, TMAX)


def verify(rows, out):
    """The exact checker: 1 if the digits of the answer, read aloud, give exactly the words of the question."""
    L, r = answer_len(out), np.zeros(len(out))
    for b, row in enumerate(rows):
        dg = out[b, : L[b] - 1]
        if out[b, L[b] - 1] != EOS or len(dg) == 0 or (dg[0] == 0 and len(dg) > 1):
            continue
        r[b] = float(read(int("".join(map(str, dg)))) == read(int(row)))
    return r


def proxy(rows, out):
    """The sloppy grader: the nonzero digits, in order, match the words for nonzero digits; the first digit is
    nonzero."""
    L, r = answer_len(out), np.zeros(len(out))
    for b, row in enumerate(rows):
        dg = out[b, : L[b] - 1]
        if out[b, L[b] - 1] != EOS or len(dg) == 0 or dg[0] == 0:
            continue
        r[b] = float([v for v in dg if v != 0] == [NZ[w] for w in read(int(row)) if w in NZ])
    return r


def pad(answers):
    """Answers (strings of digits) → a token array (n, 13), padded with ⟨end⟩."""
    return np.array([tokens(a) + [EOS] * (TMAX - len(a) - 1) for a in answers])


tries = ["150", "15000", "1500", "105", "15", "1050"]
print("question: “một trăm năm mươi” (one hundred fifty)")
for a, ok, px in zip(tries, verify(["150"] * 6, pad(tries)), proxy(["150"] * 6, pad(tries))):
    print(f"  {a:>6}: checker {int(ok)} · sloppy grader {int(px)}")
assert verify(["150"], pad(["15000"]))[0] == 0 and proxy(["150"], pad(["15000"]))[0] == 1

The sloppy grader also accepts 15000, 1500, 105, 15 and 1050 for “một trăm năm mươi” (one hundred fifty). How many
answers does it accept for one question? The nonzero digits come in a fixed order, the first digit is the first
nonzero digit, and the zeros can go anywhere. The cell below lists every answer it accepts, with at most 12 digits.

In [ ]:
def proxy_set(row):
    """Every answer (≤ 12 digits) that the sloppy grader accepts for the question of the number row."""
    want = [NZ[w] for w in read(int(row)) if w in NZ]
    out = []
    for length in range(len(want), 13):
        for pos in combinations(range(1, length), len(want) - 1):  # positions of the nonzero digits after the first
            dg = [0] * length
            dg[0] = want[0]
            for k, j in enumerate(pos):
                dg[j] = want[k + 1]
            out.append("".join(map(str, dg)))
    return out


for row in ("150", "5200", "7090400"):
    acc = proxy_set(row)
    m = sum(w in NZ for w in read(int(row)))
    print(
        f"{row:>8}: {m} nonzero digits → the sloppy grader accepts {fmt(len(acc), 0)} answers (C(12, {m}) = "
        f"{fmt(math.comb(12, m), 0)}); only 1 is correct"
    )
    assert len(acc) == math.comb(12, m) and row in acc
    assert proxy([row] * len(acc), pad(acc)).all() and verify([row] * len(acc), pad(acc)).sum() == 1

## 5 · The three post-trained number-reading models

The model is Lesson 16's GRU encoder–decoder with additive attention (H = 64). The demo uses three versions of it,
all with seed 0:
- **SFT-240:** supervised learning (cross-entropy) on only 240 labeled numbers, 20 for each length; this is
  $\pi_{\text{ref}}$;
- **RL, checker:** from SFT-240, 1,000 steps of reinforcement learning with the reward of `verify()`, $\beta = 0$;
- **RL, sloppy grader:** the same, but the reward is `proxy()`.

Reinforcement learning here uses no labels at all: each step draws 16 new numbers, the model reads them itself, and
the grader scores the answers. The cell below unpacks the float16 weights (as the demo does) and counts the
parameters.

In [ ]:
SRC = ["<pad>", *D, "mốt", "tư", "lăm", "mười", "mươi", "trăm", "linh", "nghìn", "triệu", "tỷ"]  # id 0 is the padding
W2I = {w: i for i, w in enumerate(SRC)}
assert shipped["src"] == SRC


def unpack(model):
    """base64-encoded float16 → float64 arrays, as in the demo."""
    return {
        k: np.frombuffer(base64.b64decode(v["f16"]), "<f2").astype(np.float64).reshape(v["shape"])
        for k, v in model.items()
    }


MODELS = {name: unpack(m) for name, m in shipped["models"].items()}
NAME = {"sft": "SFT-240", "rl": "RL, checker", "proxy": "RL, sloppy grader"}
n_params = sum(v.size for v in MODELS["sft"].values())
print({k: v.shape for k, v in MODELS["sft"].items()})
print(f"{fmt(n_params, 0)} parameters per model")
assert n_params == runs["setup"]["params"] == 53259


def questions(rows):
    """Numbers (strings of digits) → the word ids of the question X (padded with 0) and the mask M (1 = a word)."""
    ids = [[W2I[w] for w in read(int(r))] for r in rows]
    X = np.zeros((len(ids), max(map(len, ids))), int)
    for i, s in enumerate(ids):
        X[i, : len(s)] = s
    return X, (X > 0).astype(float)


def qset(base, n):
    """n numbers for each length 1–12 from default_rng(base + d), as in Lesson 16: validation 1000 + d, test 2000 + d."""
    rows = ["".join(map(str, r)) for d in range(1, 13) for r in sample(np.random.default_rng(base + d), d, n)]
    return rows, *questions(rows)


TEST, VAL = qset(2000, 100), qset(1000, 50)
digits = np.array([len(r) for r in TEST[0]])
print(f"test set: {fmt(len(TEST[0]), 0)} numbers · validation set: {fmt(len(VAL[0]), 0)} numbers")

The equations are Lesson 16's, in the row convention $z = xW + b$. This time the GRU function also returns the
intermediate values (`cache`), because section 7 needs to backpropagate through them.

In [ ]:
def sig(v):
    return 0.5 * (1 + np.tanh(0.5 * v))  # = 1 / (1 + e^(−v)), never overflows


def gru(ax, h, U):
    """One GRU step; ax = xW + b of all three parts (reset gate r, update gate z, candidate n), concatenated by
    column."""
    H = h.shape[1]
    rz = sig(ax[:, : 2 * H] + h @ U[:, : 2 * H])
    r, z = rz[:, :H], rz[:, H:]
    n = np.tanh(ax[:, 2 * H :] + (r * h) @ U[:, 2 * H :])
    return z * h + (1 - z) * n, (h, r, z, r * h, n)


def encode(p, X, M):
    """The GRU encoder: the states Hs (B, S, H), the final state h_T and the cache of each step."""
    h = np.zeros((X.shape[0], p["Ue"].shape[0]))
    Hs, caches = np.zeros((*X.shape, h.shape[1])), []
    for j in range(X.shape[1]):
        m = M[:, j : j + 1]
        hn, c = gru(p["Ex"][X[:, j]] + p["be"], h, p["Ue"])
        h = m * hn + (1 - m) * h  # the padding (M = 0) keeps h unchanged
        Hs[:, j] = h
        caches.append(c)
    return Hs, h, caches


def attend(p, s, Hs, keys, M):
    """Additive attention: e_j = tanh(s W_a + h_j U_a) · v_a, softmax, context c = Σ α_j h_j. keys = Hs @ U_a."""
    t = np.tanh(keys + (s @ p["Wa"])[:, None, :])
    e = np.where(M > 0, t @ p["va"], -np.inf)
    a = np.exp(e - e.max(1, keepdims=True))
    a /= a.sum(1, keepdims=True)
    return np.einsum("bs,bsh->bh", a, Hs), (a, t)


def logits(p, s, y, Hs, keys, M):
    """One decoding step after token y: the new state, the logits of the 11 outputs, and the cache."""
    c, acache = attend(p, s, Hs, keys, M)
    sn, gcache = gru(p["Ey"][y] + c @ p["Wc"] + p["bd"], s, p["Ud"])
    return sn, sn @ p["Wo"] + c @ p["Wco"] + p["bo"], (s, c, acache, gcache, sn)


def generate(p, X, M, U=None, rep=1):
    """Write the answer, at most 13 tokens. U = None: greedy (the output with the largest logit). Otherwise U (13, B·rep)
    holds uniform numbers in [0, 1): at step i, row b picks the first output whose cumulative probability exceeds
    U[i, b], as in the demo and the reel. rep: the number of answers per question (the encoder runs only once per
    question)."""
    Hs, h_T, _ = encode(p, X, M)
    keys = Hs @ p["Ua"]
    q = np.repeat(np.arange(len(X)), rep)  # the question of each row
    out = np.full((len(q), TMAX), EOS)
    act, y, s = np.arange(len(q)), np.full(len(q), BOS), h_T[q]  # the rows that haven't written ⟨end⟩ yet
    for i in range(TMAX):
        qa = q[act]
        s, z, _ = logits(p, s, y, Hs[qa], keys[qa], M[qa])
        if U is None:
            y = z.argmax(1)
        else:
            pr = np.exp(z - z.max(1, keepdims=True))
            pr /= pr.sum(1, keepdims=True)
            y = np.minimum((pr.cumsum(1) < U[i, act][:, None]).sum(1), N_OUT - 1)
        out[act, i] = y
        keep = y != EOS
        act, y, s = act[keep], y[keep], s[keep]
        if not len(act):
            break
    return out


def text(out):
    """Tokens → a string of digits (“…” if there is no ⟨end⟩)."""
    L = answer_len(out)
    return ["".join(map(str, o[: n - 1])) if o[n - 1] == EOS else "".join(map(str, o)) + "…" for o, n in zip(out, L)]

Run all three models (greedy) on the 1,200 test numbers, 100 for each length, and grade them with both graders. The
cell also compares with the seed-0 results the runs saved, to make sure this numpy version is exactly the demo's
model.

### 🤔 Predict first
The sloppy grader accepts every correct answer, plus many wrong ones. Will the model trained by reinforcement
learning against it read **more**, **as many**, or **fewer** numbers correctly than the original SFT-240? And
compared with the model trained against the exact checker?

In [ ]:
rows, X, M = TEST
STORED = {  # the saved test results of exactly these three models (seed 0)
    "sft": runs["sft"]["240"]["seeds"][0]["test"],
    "rl": runs["rl"]["b0.0"]["seeds"][0]["test"],
    "proxy": runs["rl"]["proxy0.0"]["seeds"][0]["test"],
}
greedy = {}
for name, p in MODELS.items():
    out = generate(p, X, M)
    greedy[name] = {"ok": verify(rows, out), "px": proxy(rows, out), "text": text(out)}
    g = greedy[name]
    print(f"{NAME[name]:>20}: reads {pct(g['ok'].mean())} correctly · the sloppy grader gives {pct(g['px'].mean())}")
    for key, k in (("ok", "greedy"), ("px", "greedyProxy")):
        assert g[key].sum() == round(len(rows) * STORED[name][k]), "differs from the demo: check the model again"

From 240 labeled examples, SFT reads 55.50% of the test numbers correctly (section 6 shows that it reads all 240
numbers it learned correctly). Reinforcement learning with the checker raises this to 78.17% without any extra
labels. Training against the sloppy grader raises the grader's score to 92.00%, but the true reading accuracy is only
59.67%: slightly better than SFT on this seed, far behind the model trained against the checker, and section 7 shows
that on 2 of the 3 seeds it even falls below SFT. The reason: the reward doesn't distinguish the correct answer from
the wrong answers the grader accepts, so reinforcement learning only pushes probability toward *that whole group*,
and whichever answers are easy to write gain. The cell below prints the three models' answers to a few questions,
with a label: “correct”, “sloppy” (only the sloppy grader accepts it) or “wrong”.

In [ ]:
hacked = (greedy["proxy"]["px"] == 1) & (greedy["proxy"]["ok"] == 0)
picks = [int(np.flatnonzero(hacked & (digits == d))[0]) for d in (3, 5, 8, 11)]
for i in picks:
    marks = []
    for name in MODELS:
        g = greedy[name]
        tag = "correct" if g["ok"][i] else ("sloppy" if g["px"][i] else "wrong")
        marks.append(f"{NAME[name]}: {g['text'][i]:>12} {tag:<7}")
    print(f"{rows[i]:>12} “{' '.join(read(int(rows[i])))}”")
    print("              " + " · ".join(marks))

The model trained against the sloppy grader has learned exactly the grader's loophole: it adds or drops zeros. The
cell below counts, among the greedy answers that only the sloppy grader accepts, how many are **longer** (extra
zeros) and how many are **shorter** (missing zeros) than the correct answer, by the number of digits in the question.

In [ ]:
for name in MODELS:
    g = greedy[name]
    fake = (g["px"] == 1) & (g["ok"] == 0)
    dlen = np.array([len(a) - len(r) for a, r in zip(g["text"], rows)])  # answer length − length of the correct answer
    print(
        f"{NAME[name]:>20}: {fake.sum():>3} accepted only by the sloppy grader · extra 0s: {(fake & (dlen > 0)).sum():>3}"
        f" · missing 0s: {(fake & (dlen < 0)).sum():>3} · misplaced 0s: {(fake & (dlen == 0)).sum():>2}"
    )
g = greedy["proxy"]
fake = (g["px"] == 1) & (g["ok"] == 0)
dlen = np.array([len(a) - len(r) for a, r in zip(g["text"], rows)])  # answer length − length of the correct answer
print("RL, sloppy grader, by number of digits d = 1…12 (out of 100 questions per d):")
for label, m in (
    ("truly correct", g["ok"] == 1),
    ("extra 0s", fake & (dlen > 0)),
    ("missing 0s", fake & (dlen < 0)),
    ("misplaced 0s", fake & (dlen == 0)),
):
    print(f"  {label:>13}:", [int(m[digits == d].sum()) for d in range(1, 13)])

For 3–6-digit numbers, the model trained against the sloppy grader often *adds* a 0 (74508 → 745080). From 7 digits
on, it mostly *drops* zeros (48201456040 → 4820145604): reading an 11-digit number correctly is hard, while
writing the nonzero digits in order and dropping a few zeros is much easier and still gets rewarded. The model isn't
“cheating” on purpose; it only maximizes exactly the number we give it.

## 6 · The exact probability of an answer

An answer $y = (y_1, \dots, y_L)$ is a sequence of tokens, and the model gives the probability of the whole sequence
as the product of the per-step probabilities: $\pi(y \mid x) = \prod_t \pi(y_t \mid x, y_{<t})$. To compute it, we
let the decoder read that very answer (teacher forcing, as in SFT) and add up $\log \pi(y_t \mid \dots)$ over the
tokens up to ⟨end⟩. The function `forward` below does that and keeps the cache for backpropagation in section 7.

In [ ]:
def softmax(z):
    a = np.exp(z - z.max(-1, keepdims=True))
    return a / a.sum(-1, keepdims=True)


def forward(p, X, M, Y):
    """Teacher forcing: the probabilities (B, T, 11) of every next token as the decoder reads the tokens of Y, and the
    cache."""
    Hs, h_T, ecache = encode(p, X, M)
    keys = Hs @ p["Ua"]
    Yin = np.concatenate([np.full((len(Y), 1), BOS), Y[:, :-1]], 1)  # the input of step i is token i − 1
    s, probs, dcache = h_T, [], []
    for i in range(Y.shape[1]):
        s, z, cache = logits(p, s, Yin[:, i], Hs, keys, M)
        probs.append(softmax(z))
        dcache.append(cache)
    return np.stack(probs, 1), (X, M, Yin, Hs, ecache, dcache)


def seq_logp(P, Y, L):
    """log π(y) = Σ_{t < L} log P[t, y_t] for each row, and the mask of the positions counted."""
    mask = np.arange(Y.shape[1])[None] < L[:, None]
    lp = np.log(np.take_along_axis(P, Y[:, :, None], 2)[:, :, 0])
    return (lp * mask).sum(1), mask


def logprob(p, rows, answers):
    """The exact log π(answer | question) for each pair (the number in the question, the answer), each a string of
    digits."""
    X, M = questions(rows)
    Y = pad(answers)
    return seq_logp(forward(p, X, M, Y)[0], Y, np.array([len(a) + 1 for a in answers]))[0]

Take the question “năm nghìn hai trăm” (five thousand two hundred; the default question on the demo's card 2). The
cell below lets SFT-240 write 8 answers by sampling ($T = 1$, seed 1), then computes the exact probability of the 4
answers it writes most often, and compares it with the probability saved in the demo's file.

In [ ]:
q5200 = next(c for c in cands["questions"] if c["digits"] == "5200")
rng = np.random.default_rng(1)
X1, M1 = questions(["5200"])
samples8 = text(generate(MODELS["sft"], X1, M1, rng.random((TMAX, 8)), rep=8))
print("8 samples from SFT-240:", samples8, f"· {samples8.count('5200')}/8 samples correct")
top = [c["y"] for c in q5200["cands"]][:4]
lp = logprob(MODELS["sft"], ["5200"] * len(top), top)
for a, v, c in zip(top, lp, q5200["cands"]):
    tag = "correct" if verify(["5200"], pad([a]))[0] else ("sloppy" if proxy(["5200"], pad([a]))[0] else "wrong")
    print(f"  {a:>5} ({tag:>7}): log π = {fmt(v, 3):>7} · π = {pct(math.exp(v))}")
    assert abs(math.exp(v) - c["p"]["sft"]) < 1e-6 * max(1, c["p"]["sft"]) + 1e-9, "differs from the demo"

SFT-240 writes “520” with probability 70.00% and the correct answer “5200” with only 29.28%: it *knows* the answer,
but the wrong one is preferred. The sloppy grader accepts both. On this seed 3/8 samples are “5200”; with only 8
samples, this count varies a lot around 29.28% (change the seed to see).

The checker accepts exactly **one** answer per question (the reading of a number is unique), so the probability of
the correct answer $\pi(\text{correct} \mid x)$ is also the **expected score** of a sampled answer, computed exactly,
with no sampling. The cell below computes it for all 1,200 test questions and the three models, then compares with
the results file.

In [ ]:
pc = {name: np.exp(logprob(p, rows, rows)) for name, p in MODELS.items()}  # the correct answer for the number r is r
for name in MODELS:
    print(f"{NAME[name]:>20}: expected score of one sample = mean π(correct) = {pct(pc[name].mean())}")
    assert abs(pc[name].mean() - STORED[name]["sample"]) < 1e-4, "differs from the demo"
ref_pc = np.array(runs["ref"]["pc"])
assert np.allclose(pc["sft"], ref_pc, rtol=1e-4, atol=1e-6), "differs from the demo"
print(
    f"{(pc['sft'] < 1e-6).sum()} questions where SFT-240 gives the correct answer a probability below one in a million"
)

SFT is minimizing $-\log \pi(y^{\text{example}} \mid x)$ on the example pairs: the same `logprob` function as above,
with $y$ the labeled answer. The cell below rebuilds the 240 numbers SFT-240 learned (20 for each length, drawn from
`default_rng(10_000)` as the reel's `rl.sft` does), then measures the mean cross-entropy and the reading accuracy on
those very numbers.

In [ ]:
rng = np.random.default_rng(10_000 + 0)  # seed 0
sft_rows = ["".join(map(str, r)) for d in range(1, 13) for r in sample(rng, d, 20)]
ce = -logprob(MODELS["sft"], sft_rows, sft_rows)
seen = verify(sft_rows, generate(MODELS["sft"], *questions(sft_rows)))
print(
    f"240 learned numbers: mean cross-entropy {fmt(ce.mean(), 3)} nats per answer · reads {pct(seen.mean())} correctly"
)
print(
    f"1,200 test numbers:  mean cross-entropy {fmt(-np.log(pc['sft']).mean(), 3)} nats per answer · reads "
    f"{pct(greedy['sft']['ok'].mean())} correctly"
)
assert seen.all()

SFT-240 knows the 240 numbers it learned by heart but reads only about half of new numbers correctly: 240 examples
are too few to learn all the reading rules (Lesson 08 calls this overfitting). Adding labels is one way out; another,
which the rest of the lesson uses, is to let the model practice on its own on questions *without labels* and let the
checker grade.

## 7 · GRPO-style reinforcement learning, simplified

### The objective and its gradient
Let $\pi_\theta$ be the model being trained (the policy) and $\pi_{\text{ref}}$ the SFT version, kept frozen. The
objective of RLHF (InstructGPT) and of this lesson is

$$J(\theta) = \mathbb{E}_{y \sim \pi_\theta}\big[r(x, y)\big] - \beta\, D_{\mathrm{KL}}\big(\pi_\theta \,\|\, \pi_{\text{ref}}\big).$$

We can't differentiate through sampling, but there is the identity $\nabla_\theta \mathbb{E}_{y \sim \pi_\theta}[f(y)] =
\mathbb{E}_{y \sim \pi_\theta}[f(y)\, \nabla_\theta \log \pi_\theta(y)]$ (REINFORCE, Williams 1992). Applying it to both
terms (the extra term from the KL has expectation 0, because $\mathbb{E}[\nabla \log \pi_\theta] = 0$):

$$\nabla_\theta J = \mathbb{E}_{y \sim \pi_\theta}\big[R(y)\, \nabla_\theta \log \pi_\theta(y)\big], \qquad R(y) = r(y) - \beta \log \frac{\pi_\theta(y)}{\pi_{\text{ref}}(y)}.$$

The KL penalty is just a deduction from the reward of each answer: an answer the model already prefers much more
than $\pi_{\text{ref}}$ does gets a large deduction. InstructGPT deducts exactly this amount at each token, with
$\beta = 0.02$.

### The within-group baseline
For each question we take $G = 8$ answers. Subtracting from $R_i$ a number $b$ that doesn't depend on $y_i$ leaves
the expected gradient unchanged but reduces its variance. We take the mean of **the other 7 answers** to the same
question (leave-one-out, as in the RLOO of Ahmadian et al., 2024):

$$A_i = R_i - \frac{1}{G - 1} \sum_{j \neq i} R_j.$$

GRPO (Shao et al., 2024; DeepSeek-R1) also uses the group in place of PPO's value network, but it subtracts the mean
of the whole group and then divides by the group's standard deviation, adds PPO's clipping of the ratio
$\pi_\theta / \pi_{\text{old}}$, and puts the KL penalty directly into the loss instead of deducting it from the
reward. Here each batch is used for only one update, so no clipping is needed; that is why the course calls it
**GRPO-style, simplified**.

### From formula to code
The gradient above is the gradient of the surrogate loss $\mathcal{L} = -\frac{1}{N}\sum_i A_i \log \pi_\theta(y_i)$
when $A_i$ is treated as a constant, with $N = 16 \cdot 8$ answers per step. With respect to the logits at step $t$,
the derivative is $\frac{A_i}{N}\,(p_t - \text{onehot}(y_t))$: exactly the cross-entropy of SFT, only multiplied by
$A_i$. $A_i > 0$ pushes the probability of that answer up, $A_i < 0$ pushes it down. The cell below writes
backpropagation through the decoder with attention and through the GRU encoder (like the reel's `rl.tf`); the cell
after it checks it with finite differences.

In [ ]:
def gru_back(dh_new, cache, U, dU):
    """Backpropagate through one GRU step: the gradients with respect to ax and to the old state h; adds the gradient
    of U into dU."""
    h, r, z, rh, n = cache
    H = h.shape[1]
    dz, dn, dh = dh_new * (h - n), dh_new * (1 - z), dh_new * z  # h_new = z·h + (1 − z)·n
    dan = dn * (1 - n**2)  # through tanh
    drh = dan @ U[:, 2 * H :].T
    dh += drh * r
    darz = np.concatenate([drh * h * r * (1 - r), dz * z * (1 - z)], 1)  # through the two sigmoids
    dU[:, : 2 * H] += h.T @ darz
    dU[:, 2 * H :] += rh.T @ dan
    dh += darz @ U[:, : 2 * H].T
    return np.concatenate([darz, dan], 1), dh


def backward_decoder(p, g, cache, dZ):
    """The decoder, from the last step back to the first. Returns the gradients with respect to Hs, to keys = Hs @ U_a,
    and to h_T."""
    _, _, Yin, Hs, _, dcache = cache
    dHs, dkeys, ds = np.zeros_like(Hs), np.zeros((*Hs.shape[:2], p["Ua"].shape[1])), np.zeros_like(Hs[:, 0])
    for i in reversed(range(len(dcache))):
        s, c, (a, t), gcache, sn = dcache[i]
        dz = dZ[:, i]
        g["Wo"] += sn.T @ dz
        g["Wco"] += c.T @ dz
        g["bo"] += dz.sum(0)
        dax, ds = gru_back(ds + dz @ p["Wo"].T, gcache, p["Ud"], g["Ud"])
        np.add.at(g["Ey"], Yin[:, i], dax)
        g["bd"] += dax.sum(0)
        g["Wc"] += c.T @ dax
        dc = dz @ p["Wco"].T + dax @ p["Wc"].T
        da = np.einsum("bh,bsh->bs", dc, Hs)  # c = Σ a_j h_j
        dHs += a[:, :, None] * dc[:, None, :]
        de = a * (da - (a * da).sum(1, keepdims=True))  # through softmax
        g["va"] += np.einsum("bs,bsa->a", de, t)
        dt = de[:, :, None] * p["va"] * (1 - t**2)  # e_j = tanh(keys_j + s W_a) · v_a
        dkeys += dt
        g["Wa"] += s.T @ dt.sum(1)
        ds = ds + dt.sum(1) @ p["Wa"].T  # the query is the state before step i
    return dHs, dkeys, ds


def backward(p, cache, dZ):
    """The gradients of every parameter, given dL/dlogits dZ (B, T, 11) at every step."""
    X, M, _, Hs, ecache, _ = cache
    g = {k: np.zeros_like(v) for k, v in p.items()}
    dHs, dkeys, dh = backward_decoder(p, g, cache, dZ)
    g["Ua"] += np.einsum("bsh,bsa->ha", Hs, dkeys)
    dHs += dkeys @ p["Ua"].T
    for j in reversed(range(X.shape[1])):  # the encoder; h_T is the decoder's initial state
        dh = dh + dHs[:, j]
        m = M[:, j : j + 1]
        dax, dh_prev = gru_back(m * dh, ecache[j], p["Ue"], g["Ue"])
        np.add.at(g["Ex"], X[:, j], dax)
        g["be"] += dax.sum(0)
        dh = dh_prev + (1 - m) * dh  # the padding only passes h through
    return g


def surrogate_grad(P, Y, L, A):
    """dL/dlogits of L = −(1/N) Σ_i A_i log π(y_i): A_i (p_t − onehot(y_t)) / N at the positions counted."""
    mask = np.arange(Y.shape[1])[None] < L[:, None]
    return A[:, None, None] * (P - np.eye(N_OUT)[Y]) * mask[:, :, None] / len(Y)

A finite-difference check on the SFT-240 model itself: 3 questions, 3 random answers of lengths 4, 2, 3 and the
advantages $A = (0.7,\ -1.3,\ 0.4)$. For each matrix, we compare the 3 entries with the largest gradient with
$(\mathcal{L}(\theta + \varepsilon) - \mathcal{L}(\theta - \varepsilon)) / 2\varepsilon$.

In [ ]:
rng = np.random.default_rng(0)
Xf, Mf = questions(["150", "2005", "1000001"])
Yf, Lf, Af = rng.integers(0, N_OUT, (3, 4)), np.array([4, 2, 3]), np.array([0.7, -1.3, 0.4])
pf = {k: v.copy() for k, v in MODELS["sft"].items()}


def surrogate(q):
    return -(Af * seq_logp(forward(q, Xf, Mf, Yf)[0], Yf, Lf)[0]).sum() / 3


P_f, cache_f = forward(pf, Xf, Mf, Yf)
g_f = backward(pf, cache_f, surrogate_grad(P_f, Yf, Lf, Af))
worst = 0.0
for k, w in pf.items():
    for flat in np.argsort(-np.abs(g_f[k]).ravel())[:3]:
        idx = np.unravel_index(flat, w.shape)
        old = w[idx]
        w[idx] = old + 1e-5
        up = surrogate(pf)
        w[idx] = old - 1e-5
        down = surrogate(pf)
        w[idx] = old
        num = (up - down) / 2e-5
        worst = max(worst, abs(num - g_f[k][idx]) / max(1e-4, abs(num) + abs(g_f[k][idx])))
print(f"{3 * len(pf)} entries, largest relative error {sci(worst)}")
assert worst < 1e-6

The hand-written gradient matches the finite differences. Now put it together into one reinforcement-learning step,
like the reel's `rl.rl`:
1. draw $Q = 16$ new numbers (without labels) from `default_rng(50_000 + seed)`;
2. let the model write $G = 8$ answers for each ($T = 1$);
3. grade them with the checker, compute $R_i = r_i - \beta \log(\pi_\theta(y_i)/\pi_{\text{ref}}(y_i))$ and the
   advantage $A_i$;
4. take one Adam step ($\eta = 10^{-3}$, gradient norm clipped at 1) on the surrogate loss.

The function `evaluate` measures, on the 600 validation numbers, exactly the columns the reel saves every 100 steps:
reading accuracy (greedy), the sloppy grader's score, the mean $\pi(\text{correct})$, and the KL to $\pi_{\text{ref}}$
(nats). The KL of a whole answer is estimated by taking one sample per question and adding up the **exact** KL of
each token along that sample:
$\sum_t \sum_v \pi_\theta(v \mid \dots) \log \frac{\pi_\theta(v \mid \dots)}{\pi_{\text{ref}}(v \mid \dots)}$.

In [ ]:
class Adam:
    """Adam (Lesson 07) with gradient-norm clipping, as in the reel."""

    def __init__(self, p):
        self.m = {k: np.zeros_like(v) for k, v in p.items()}
        self.v = {k: np.zeros_like(v) for k, v in p.items()}
        self.t = 0

    def step(self, p, g, lr, clip=1.0):
        self.t += 1
        norm = math.sqrt(sum((x**2).sum() for x in g.values()))
        scale = min(1.0, clip / norm)
        for k in p:
            self.m[k] = 0.9 * self.m[k] + 0.1 * scale * g[k]
            self.v[k] = 0.999 * self.v[k] + 0.001 * (scale * g[k]) ** 2
            m_hat, v_hat = self.m[k] / (1 - 0.9**self.t), self.v[k] / (1 - 0.999**self.t)
            p[k] -= lr * m_hat / (np.sqrt(v_hat) + 1e-8)


def rl_step(p, ref, opt, rng, beta, reward=verify, Q=16, G=8, lr=1e-3):
    """One step of GRPO-style reinforcement learning, simplified. Returns the grader's rewards (Q, G)."""
    rows = ["".join(map(str, sample(rng, int(d), 1)[0])) for d in rng.integers(1, 13, Q)]
    Xq, Mq = questions(rows)
    Y = generate(p, Xq, Mq, rng.random((TMAX, Q * G)), rep=G)  # G answers for each question
    Xg, Mg, rows_g = np.repeat(Xq, G, 0), np.repeat(Mq, G, 0), list(np.repeat(rows, G))
    r, L = reward(rows_g, Y), answer_len(Y)
    P, cache = forward(p, Xg, Mg, Y)
    lp = seq_logp(P, Y, L)[0]
    lq = seq_logp(forward(ref, Xg, Mg, Y)[0], Y, L)[0] if beta else lp  # β = 0: π_ref isn't needed
    R = (r - beta * (lp - lq)).reshape(Q, G)
    A = R - (R.sum(1, keepdims=True) - R) / (G - 1)  # baseline: the mean of the other 7 answers
    opt.step(p, backward(p, cache, surrogate_grad(P, Y, L, A.ravel())), lr)
    return r.reshape(Q, G)


def kl_sampled(p, ref, X, M, seed):
    """KL(π_θ‖π_ref) of a whole answer (nats), averaged over one sample per question."""
    Y = generate(p, X, M, np.random.default_rng(seed).random((TMAX, len(X))))
    Pp, Pq = forward(p, X, M, Y)[0], forward(ref, X, M, Y)[0]
    per_token = (Pp * np.log(Pp / Pq)).sum(-1)
    return float((per_token * (np.arange(TMAX)[None] < answer_len(Y)[:, None])).sum(1).mean())


def evaluate(p, ref, data, seed):
    rows_v, Xv, Mv = data
    out = generate(p, Xv, Mv)
    return {
        "correct": verify(rows_v, out).mean(),
        "sloppy": proxy(rows_v, out).mean(),
        "π(correct)": np.exp(logprob(p, rows_v, rows_v)).mean(),
        "KL": kl_sampled(p, ref, Xv, Mv, seed),
    }

### 🤔 Predict first
We run 100 steps with $\beta = 0.1$ from SFT-240. At each step, the checker grades $16 \times 8$ answers. What
percentage of the **groups** of 8 answers (to the same question) contain both correct and wrong answers, that is,
carry a signal to learn from: close to 100%, about half, or under a quarter?

In [ ]:
ref = MODELS["sft"]
policy = {k: v.copy() for k, v in ref.items()}
opt, rng = Adam(policy), np.random.default_rng(50_000 + 0)  # seed 0, as in the reel
mixed = []
for step in range(100):
    rg = rl_step(policy, ref, opt, rng, beta=0.1)
    mixed.append(((rg.max(1) - rg.min(1)) > 0).mean())  # groups with both correct and wrong answers
before, after = evaluate(ref, ref, VAL, 0), evaluate(policy, ref, VAL, 100)
for key in before:
    f = (lambda v: fmt(v, 3)) if key == "KL" else pct
    print(f"{key:>10}: SFT-240 {f(before[key]):>7} → after 100 steps {f(after[key]):>7}")
print(
    f"groups of 8 with both correct and wrong answers: {pct(np.mean(mixed), 1)}"
    f" ({sum(round(16 * m) for m in mixed)} of 1,600 groups)"
)
stored100 = runs["rl"]["b0.1"]["seeds"][0]["curve"][1]  # [step, correct, sloppy, π(correct), KL, mixed groups]
# this is training: on another machine, rounding errors can change a few samples, so compare with a tolerance of a few
# questions
assert (
    stored100[0] == 100 and abs(after["correct"] - stored100[1]) < 5e-3 and abs(after["sloppy"] - stored100[2]) < 5e-3
)
assert abs(after["π(correct)"] - stored100[3]) < 5e-3 and abs(after["KL"] - stored100[4]) < 0.05, (
    "differs from the reel"
)
assert abs(np.mean(mixed) - stored100[5]) < 5e-3, "differs from the reel"

Only about a quarter of the groups contain both correct and wrong answers. In the other groups, all 8 answers are
correct (easy questions) or all are wrong (questions that are too hard), the $r_i$ are equal, so $A_i$ is left with
only the differences in the KL penalty: the checker's reward teaches nothing there. Dividing by the standard
deviation as GRPO does doesn't rescue it either, because $0/0$ is set to 0. This is why some systems that train
reasoning drop the all-correct and all-wrong groups when building a batch and sample other questions instead (the
dynamic sampling of DAPO, Yu et al., 2025).

After 100 steps (of the reel's 1,000 in total), the reading accuracy on the validation set has risen and the model
has drifted about 0.89 nats from $\pi_{\text{ref}}$. The printed numbers match the “step 100” row the reel saved for
exactly this run (same seed, same sequence of random numbers; on another machine they may be off by a question or
two because of rounding): the numpy version above is exactly the algorithm that produced the demo's models.

### All the runs
1,000 steps for each configuration and 3 seeds take too long for a notebook. The cell below reads the test results
of every run from the demo's file (1,200 test numbers): reading accuracy (greedy), the sloppy grader's score,
correct with one sample (mean $\pi(\text{correct})$), the KL to the SFT version, and the fraction of groups with both
correct and wrong answers. Each cell is the lowest–highest range over 3 seeds.

In [ ]:
ROWS = [  # (label, key in the results file)
    ("SFT-240", ("sft", "240")),
    ("SFT-960", ("sft", "960")),
    ("SFT-3,840", ("sft", "3840")),
    ("SFT-240 + RL, β = 0", ("rl", "b0.0")),
    ("SFT-240 + RL, β = 0.03", ("rl", "b0.03")),
    ("SFT-240 + RL, β = 0.1", ("rl", "b0.1")),
    ("SFT-240 + RL, β = 0.3", ("rl", "b0.3")),
    ("… divide by std, β = 0", ("rl", "norm0.0")),
    ("SFT-960 + RL, β = 0", ("rl", "n960b0.0")),
    ("… sloppy grader, β = 0", ("rl", "proxy0.0")),
    ("… sloppy grader, β = 0.1", ("rl", "proxy0.1")),
]
TABLE = {}
print(f"{'':>28} {'correct':>15} {'sloppy score':>15} {'correct, 1 sample':>17} {'KL (nats)':>11} {'mixed groups':>13}")
for label, (kind, key) in ROWS:
    seeds = runs[kind][key]["seeds"]
    t = [s["test"] for s in seeds]
    row = {k: [x[k] for x in t] for k in ("greedy", "greedyProxy", "sample")}
    row["kl"] = [x.get("kl", 0.0) for x in t]
    row["mixed"] = [s.get("mixed", float("nan")) for s in seeds]
    TABLE[key if kind == "rl" else f"sft{key}"] = row
    kl = span(row["kl"], lambda v: fmt(v, 2)) if kind == "rl" else "—"
    mx = span(row["mixed"], lambda v: pct(v, 1)) if kind == "rl" else "—"
    print(
        f"{label:>28} {span(row['greedy']):>15} {span(row['greedyProxy']):>15} {span(row['sample']):>17} {kl:>11} {mx:>13}"
    )
below = sum(a < b for a, b in zip(TABLE["proxy0.0"]["greedy"], TABLE["sft240"]["greedy"]))
print(f"trained against the sloppy grader (β = 0): {below}/3 seeds read fewer correctly than that seed's own SFT-240")

Reading the table from top to bottom:
- **Reinforcement learning with the checker** takes SFT-240 from 53.25–56.00% to 75.33–78.17% (β = 0), with no extra
  labels, but drifts 3.57–3.99 nats from the SFT version. With β = 0.1, the model drifts much less (1.19–1.96 nats)
  and is 67.58–73.25% correct.
- **More labels are still better here:** SFT on 960 numbers already reaches 85.17–86.50%, more than any
  reinforcement-learning run from SFT-240. Reinforcement learning is good at polishing a model that is already
  decent; it doesn't replace good data.
- **Dividing by the standard deviation** as GRPO does gives 68.75–77.67%, versus 75.33–78.17% when subtracting the
  mean of the other 7 answers: the two ranges overlap, within the variation between seeds.
- **The sloppy grader** pushes its own score up to 88.00–92.67%, while the true reading accuracy is only
  44.75–59.67%. Section 10 comes back to this.

The figure below plots the learning curves on the 600 validation numbers (line: mean over 3 seeds, band:
lowest–highest): on the left the reading accuracy, on the right the KL to the SFT version. Blue, round dots: β = 0;
green, triangles, dashed: β = 0.1.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.4))
for key, color, marker, ls, label in (("b0.0", BLUE, "o", "-", "β = 0"), ("b0.1", GREEN, "^", "--", "β = 0.1")):
    curves = np.array([s["curve"] for s in runs["rl"][key]["seeds"]], dtype=float)  # (seed, measurement, column)
    steps = curves[0, :, 0]
    for ax, col, scale in ((a1, 1, 100), (a2, 4, 1)):
        v = scale * curves[:, :, col]
        ax.fill_between(steps, v.min(0), v.max(0), color=color, alpha=0.18)
        ax.plot(steps, v.mean(0), marker=marker, ls=ls, color=color, ms=4, label=label)
a1.set(xlabel="reinforcement-learning step", ylabel="correct, validation set (%)", title="Reading accuracy rises…")
a2.set(
    xlabel="reinforcement-learning step", ylabel="KL(π_θ ‖ π_ref), nats", title="…and the model leaves the SFT version"
)
for ax in (a1, a2):
    ax.grid(alpha=0.3)
    ax.legend(frameon=False)
fmt_axes(fig)
plt.show()
for key, label in (("b0.0", "β = 0  "), ("b0.1", "β = 0.1")):
    kls = [c[4] for s in runs["rl"][key]["seeds"] for c in s["curve"][1:]]  # steps 100 … 1,000, 3 seeds
    print(f"{label}: KL to the SFT version at steps 100–1,000 (3 seeds): {fmt(min(kls), 2)}–{fmt(max(kls), 2)} nats")

The model leaves the SFT version within the first 100 steps. With β = 0, nothing pulls it back: the KL wanders over a
wide range, from 1.59 to 5.36 nats. With β = 0.1, the penalty holds the KL at 0.89–2.61 nats, and the reading
accuracy rises more slowly: that is the price of the leash.

## 8 · The closed-form solution of the KL-penalized problem

When the set of answers is finite, we can solve the problem of section 7 by hand. Write $\pi^*(y) = \pi_{\text{ref}}(y)\,
e^{r(y)/\beta} / Z$ with $Z = \sum_y \pi_{\text{ref}}(y)\, e^{r(y)/\beta}$. Then

$$J(\pi) = \sum_y \pi(y)\, r(y) - \beta \sum_y \pi(y) \log \frac{\pi(y)}{\pi_{\text{ref}}(y)} = -\beta \sum_y \pi(y)
\log \frac{\pi(y)}{\pi^*(y)} + \beta \log Z = -\beta\, D_{\mathrm{KL}}(\pi \,\|\, \pi^*) + \beta \log Z.$$

The KL is non-negative and equals 0 only when $\pi = \pi^*$, so $\pi^*$ is the optimal solution: take
$\pi_{\text{ref}}$, multiply each answer by $e^{r/\beta}$, then renormalize. With a large β, $\pi^*$ keeps
$\pi_{\text{ref}}$ almost unchanged; as β → 0 it piles all the probability onto the answers with the highest reward,
but **keeps the ratios** between them as they are in $\pi_{\text{ref}}$. No answer to which $\pi_{\text{ref}}$ gives
probability 0 can appear in $\pi^*$.

Take “năm nghìn hai trăm” (five thousand two hundred) again. The file `posttrain-cands.json` has the exact
probability of every common answer (the distinct answers among 512 samples from SFT-240) and two “buckets” for the
rest: the other answers the sloppy grader accepts, and the other wrong answers. The cell below computes $\pi^*$ for a
few values of β.

In [ ]:
def items(c, model="sft"):
    """Every answer to a question of card 2, then the two buckets: the probability under `model`, the reward r and the
    sloppy score."""
    labels = [x["y"] for x in c["cands"]] + ["other, the sloppy grader accepts", "other, wrong"]
    pr = [x["p"][model] for x in c["cands"]] + [c["otherProxy"][model], c["other"][model]]
    r = [x["r"] for x in c["cands"]] + [0, 0]
    px = [x["proxy"] for x in c["cands"]] + [1, 0]
    keep = [i for i, v in enumerate(pr) if v > 0]
    return [labels[i] for i in keep], *(np.array([v[i] for i in keep], float) for v in (pr, r, px))


def star(pref, r, beta):
    """π* ∝ π_ref·e^{r/β}, E_π*[r] and KL(π*‖π_ref) (nats)."""
    w = pref * np.exp((r - r.max()) / beta)  # subtract r.max() so that a small β doesn't overflow
    q = w / w.sum()
    return q, q @ r, float((q * np.log(np.where(q > 0, q / pref, 1))).sum())


lab, pref, r5, px5 = items(q5200)
print("π_ref:", " · ".join(f"{a} {pct(v)}" for a, v in zip(lab, pref)))
for beta in (3, 1, 0.3, 0.1, 0.03):
    q, er, kl = star(pref, r5, beta)
    print(f"β = {fmt(beta, 2):>4}: π*(5200) = {pct(q[lab.index('5200')]):>7} · KL(π*‖π_ref) = {fmt(kl, 3)} nats")
assert abs(pref.sum() - 1) < 1e-6 and abs(pref[lab.index("5200")] - math.exp(lp[1])) < 1e-6

π* trades reward for drift: to put 99% of the probability on the correct answer, it has to drift about 1.2 nats from
$\pi_{\text{ref}}$; the KL never exceeds $-\log \pi_{\text{ref}}(5200) = 1.23$ nats, the level of keeping only the
correct answer.

Two checks on this very table, at β = 0.1. On a finite set, the model is the softmax of the logits $\theta$; the
exact gradient of $J$ with respect to $\theta_k$ is $\pi_k\,(R_k - \mathbb{E}_\pi[R])$ with
$R = r - \beta \log(\pi/\pi_{\text{ref}})$.
1. Gradient ascent along that gradient from $\pi_{\text{ref}}$ converges to exactly $\pi^*$.
2. The REINFORCE estimate of section 7 (groups of 8 samples, the baseline being the mean of the other 7 samples) is
   unbiased: the mean over 20,000 groups matches the exact gradient, within a few standard errors.

In [ ]:
beta = 0.1
q_star = star(pref, r5, beta)[0]


def exact_grad(theta):
    pi = softmax(theta)
    R = r5 - beta * np.log(pi / pref)
    return pi * (R - pi @ R), pi, R


theta = np.log(pref).copy()
for _ in range(20_000):
    theta += 5.0 * exact_grad(theta)[0]
print(f"gradient ascent from π_ref: largest |π − π*| = {sci(np.abs(softmax(theta) - q_star).max())}")
assert np.abs(softmax(theta) - q_star).max() < 1e-4

theta0 = np.log(star(pref, r5, 0.3)[0])  # a point halfway along, where the KL penalty is nonzero
g_exact, pi0, R0 = exact_grad(theta0)
rng = np.random.default_rng(0)
Y = np.minimum((rng.random((20_000, 8, 1)) > np.cumsum(pi0)).sum(2), len(pi0) - 1)  # 20,000 groups × 8 samples
A = R0[Y] - (R0[Y].sum(1, keepdims=True) - R0[Y]) / 7
g_mc = (A[:, :, None] * (np.eye(len(pi0))[Y] - pi0)).mean(1)  # ∇ log π(y) with respect to θ = onehot(y) − π
z_score = (g_mc.mean(0) - g_exact) / (g_mc.std(0) / math.sqrt(len(g_mc)))
for a, ge, gm in zip(lab, g_exact, g_mc.mean(0)):
    print(f"  {a:>32}: exact {fmt(ge, 4):>8} · REINFORCE {fmt(gm, 4):>8}")
print(f"{fmt(len(g_mc), 0)} groups × 8 samples; largest difference: {fmt(np.abs(z_score).max(), 2)} standard errors")
assert np.abs(z_score).max() < 4

Both checks pass. Real reinforcement learning does exactly this, only over billions of possible answers, with a
neural network shared by every question, and with a gradient estimated from a few samples.

## 9 · DPO: learning straight from good–bad pairs

Classic RLHF (InstructGPT) learns a **reward model** from the pairs people chose, then runs reinforcement learning.
DPO (Rafailov et al., 2023) skips both steps by taking the log of both sides of $\pi^*$:

$$r(x, y) = \beta \log \frac{\pi^*(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x).$$

In the Bradley–Terry model, a person prefers $y_w$ to $y_l$ with probability $\sigma(r(y_w) - r(y_l))$, so only the
*difference* of the two rewards appears and $Z(x)$ cancels. Replacing $\pi^*$ with the model being trained
$\pi_\theta$, we get a classification loss on pairs:

$$\mathcal{L}_{\text{DPO}} = -\log \sigma\Big(\beta \log \frac{\pi_\theta(y_w)}{\pi_{\text{ref}}(y_w)} - \beta \log
\frac{\pi_\theta(y_l)}{\pi_{\text{ref}}(y_l)}\Big).$$

No separate reward model is needed, no sampling during training, but $\pi_{\text{ref}}$ is still needed:
$\beta \log(\pi_\theta/\pi_{\text{ref}})$ is exactly the **implicit reward** that DPO learns.

We try it on the table for “năm nghìn hai trăm”, as the demo does: draw pairs $y_1, y_2 \sim \pi_{\text{ref}}$
(redraw $y_2$ until it differs from $y_1$), then let a simulated “person” choose $y_1$ with probability
$\sigma(r(y_1) - r(y_2))$. The random number generator is the demo's mulberry32 (as in Lesson 16), so that the pairs
are the same as the pairs in the browser.

In [ ]:
def mulberry32(seed, n):
    """n uniform numbers in [0, 1) from mulberry32(seed), identical to the demo's JavaScript function."""
    a = ((seed + np.arange(1, n + 1, dtype=np.uint64) * 0x6D2B79F5) & 0xFFFFFFFF).astype(np.uint32)
    t = (a ^ (a >> 15)) * (a | 1)
    t = (t + (t ^ (t >> 7)) * (t | 61)) ^ t
    return (t ^ (t >> 14)).astype(np.float64) / 2**32


def pairs(pref, r, n, seed):
    """n preference pairs, as in the demo: W[w, l] is the number of times y_w beats y_l."""
    size = 4 * n
    while True:
        u = mulberry32(seed, size)
        pick = np.minimum(np.searchsorted(np.cumsum(pref), u, side="right"), len(pref) - 1)  # drawn from π_ref
        W, k = np.zeros((len(pref), len(pref))), 0
        try:
            for _ in range(n):
                i1, i2, k, tries = pick[k], pick[k + 1], k + 2, 1
                while i2 == i1:  # redraw y₂ (after 1,000 tries, take the next item)
                    if tries >= 1000:
                        i2 = (i1 + 1) % len(pref)
                        break
                    i2, k, tries = pick[k], k + 1, tries + 1
                win = u[k] < sig(r[i1] - r[i2])  # Bradley–Terry
                W[(i1, i2) if win else (i2, i1)] += 1
                k += 1
            return W
        except IndexError:  # the sequence isn't long enough: take one twice as long (with the same beginning)
            size *= 2


def dpo_fit(W, pref, beta, ref=True, steps=3000, lr=0.5):
    """DPO on the table: π_θ = softmax(θ), starting from π_ref. Written in terms of v = β(θ − log π_ref), the margin of a
    pair is v_w − v_l (with π_ref); without π_ref, the margin is β log π_θ(y_w)/π_θ(y_l). Full-batch gradient descent
    on −log σ(margin)."""
    off = np.zeros(len(pref)) if ref else beta * np.log(pref)
    v = np.zeros(len(pref))
    for _ in range(steps):
        margin = (v + off)[:, None] - (v + off)[None, :]
        d = W * (1 - sig(margin))  # −∂L/∂margin, for every pair (w, l)
        v -= lr * beta**2 * (d.sum(0) - d.sum(1)) / W.sum()
    return v


def implicit(v, pref, beta):
    """The fitted policy π ∝ π_ref·e^{v/β} and the implicit reward β log(π/π_ref)."""
    pi = pref * np.exp((v - v.max()) / beta)
    pi /= pi.sum()
    return beta * np.log(pi / pref), pi


W = pairs(pref, r5, 3000, 0)
print(f"3,000 pairs, seed 0: win counts (row beats column), items {lab[:4]}:")
print(W[:4, :4].astype(int))
i_w, i_l = lab.index("5200"), lab.index("520")
print(
    f"“5200” beats “520” {fmt(W[i_w, i_l], 0)} times and loses {fmt(W[i_l, i_w], 0)} times: "
    f"{pct(W[i_w, i_l] / (W[i_w, i_l] + W[i_l, i_w]))}; σ(1) = {pct(sig(1.0))}"
)

The correct answer “5200” beats “520” 2,166 times and loses 765 times, close to the Bradley–Terry ratio $\sigma(1)$
for a reward difference of 1. The cell below fits DPO ($\beta = 0.1$) with 30, 300 and 3,000 pairs, 3 seeds per size,
with and without $\pi_{\text{ref}}$ in the margin. Then it compares the implicit reward with the true reward $r$,
after subtracting from each side its mean (weighted by $\pi_{\text{ref}}$) over the answers that appear in at least
one pair, because a reward is only defined up to a constant. The “deviation” column is the $\pi_{\text{ref}}$-weighted
mean of $|$implicit reward − $r|$, on a scale where a correct answer and a wrong answer are 1 apart, as in the demo.

In [ ]:
def dpo_report(pref, r, n, seed, beta=0.1, ref=True):
    """Fit DPO on n pairs; returns the mean deviation of the implicit reward, E_π[r] and KL(π‖π_ref)."""
    W = pairs(pref, r, n, seed)
    imp, pi = implicit(dpo_fit(W, pref, beta, ref), pref, beta)
    seen = (W.sum(0) + W.sum(1)) > 0  # the answers that appear in at least one pair
    w = pref[seen].sum()
    err = np.abs((imp - (pref * imp)[seen].sum() / w) - (r - (pref * r)[seen].sum() / w))
    return (pref * err)[seen].sum() / w, pi @ r, float((pi * np.log(pi / pref)).sum())


dpo = {(n, w): [dpo_report(pref, r5, n, s, ref=w) for s in range(3)] for n in (30, 300, 3000) for w in (1, 0)}
print(f"π* (β = 0.1): correct {pct(star(pref, r5, 0.1)[1])}, KL {fmt(star(pref, r5, 0.1)[2], 3)} nats")
for (n, with_ref), res in dpo.items():
    err, er, kl = zip(*res)
    print(
        f"{fmt(n, 0):>5} pairs, {'with π_ref   ' if with_ref else 'without π_ref'}: deviation "
        f"{span(err, lambda v: fmt(v, 3)):>13} · policy correct {span(er):>15} · KL {span(kl, lambda v: fmt(v, 3))} nats"
    )
assert max(e for e, _, _ in dpo[3000, 1]) < 0.08 and min(e for _, e, _ in dpo[3000, 1]) > 0.99

With 3,000 pairs, DPO with $\pi_{\text{ref}}$ recovers $\pi^*$ closely: the policy is over 99% correct, its KL is
close to 1.2 nats like that of $\pi^*$, and the implicit reward matches $r$ with a small deviation on all 3 seeds. This
question is easy for DPO: almost every pair is “520” against “5200”, so 30 pairs are enough to choose correctly; only
the deviation varies more between seeds, because the rare answers don't appear in any pair and DPO knows nothing
about them.

Dropping $\pi_{\text{ref}}$ from the margin (the margin becomes $\beta \log \pi_\theta(y_w)/\pi_\theta(y_l)$) still
gives a good policy on this small problem, but the meaning has changed: the “implicit reward” is no longer the change
relative to the original model, and nothing keeps the model close to $\pi_{\text{ref}}$ anymore. On all 12 questions
of card 2 (3 seeds per question), the demo's results file shows how much the number of pairs matters when many
answers compete:

In [ ]:
for d in cands["dpo"]:
    print(
        f"{fmt(d['n'], 0):>5} pairs, {'with π_ref   ' if d['ref'] else 'without π_ref'}: deviation "
        f"{fmt(d['mean'][0], 3)}–{fmt(d['mean'][1], 3)} · correct {pct(d['er'][0])}–{pct(d['er'][1])}"
    )

With 30 pairs, there is a question on which the DPO policy is only 26.62% correct; with 3,000 pairs, every question
is above 99%. DPO needs enough pairs to cover the answers that matter, because it can only learn from what is in the
preference data.

## 10 · Goodhart: when a measure becomes a target

“When a measure becomes a target, it ceases to be a good measure” (Goodhart's law). Gao, Schulman & Hilton (2022)
measured this with real reward models: optimizing against a proxy reward model, the “gold” score (graded by a larger
model) rises and then falls as the KL to the initial model grows. The number-reading task lets us see the same
phenomenon exactly, because we have both the true measure (`verify`) and the broken one (`proxy`).

Start with the closed-form solution. If the reward is `proxy`, $\pi^*$ piles the probability onto **every** answer
the sloppy grader accepts, and keeps the ratios between them as in $\pi_{\text{ref}}$. For “năm nghìn hai trăm”,
$\pi_{\text{ref}}$ prefers “520” to “5200” by more than two to one, so optimizing the sloppy grader never makes the
correct answer win:

In [ ]:
for beta in (1, 0.3, 0.1, 0.03):
    q_r, q_px = star(pref, r5, beta)[0], star(pref, px5, beta)[0]
    print(
        f"β = {fmt(beta, 2):>4}: optimizing the checker → correct {pct(q_r @ r5):>7}"
        f" · optimizing the sloppy grader → sloppy score {pct(q_px @ px5):>7}, truly correct {pct(q_px @ r5):>7}"
    )
ceiling_5200 = pref[lab.index("5200")] / (pref * px5).sum()  # the correct answer's share of the sloppy-accepted answers
print(f"β → 0, sloppy grader: truly correct tends to π_ref(5200) / π_ref(sloppy-accepted) = {pct(ceiling_5200)}")

It is the same on all 1,200 test questions: optimizing the sloppy grader perfectly (β → 0) gives an accuracy equal
to the mean of $\pi_{\text{ref}}(\text{correct}) / \pi_{\text{ref}}(\text{sloppy-accepted})$. The cell below computes
it from $\pi_{\text{ref}}(\text{correct})$, computed in section 6, and the mass the sloppy grader accepts under
$\pi_{\text{ref}}$ (saved in the results file; the reel computes it exactly by walking the tree of every answer the
sloppy grader accepts).

In [ ]:
pm_ref = np.array(runs["ref"]["pm"])  # π_ref(the sloppy grader accepts the answer), for each test question
# the correct answer is always among the answers the sloppy grader accepts; the tree walk drops branches whose
# probability is below 10⁻¹⁰
assert (pm_ref >= pc["sft"] * (1 - 1e-4) - 1e-10).all()
share = np.where(pm_ref > 0, pc["sft"] / np.maximum(pm_ref, 1e-300), 0.0)  # the correct answer's share of that mass
print(f"SFT-240: π(correct) {pct(pc['sft'].mean())} · π(sloppy-accepted) {pct(pm_ref.mean())}")
print(f"optimizing the sloppy grader perfectly (β → 0): sloppy score 100%, truly correct {pct(share.mean())}")
print(
    f"optimizing the checker perfectly (β → 0): correct {pct(np.mean(pc['sft'] > 0))}, with a mean KL of"
    f" −log π_ref(correct) = {fmt(-np.log(pc['sft']).mean(), 2)} nats"
)

Even optimizing the sloppy grader perfectly is only 61.85% correct, and the real reinforcement-learning model does
worse than that. (Optimizing the checker perfectly is 100.00% correct, at the cost of drifting 5.44 nats on average.)
The figure below plots the three runs trained against the sloppy grader with β = 0, on the 600 validation numbers
(line: mean over 3 seeds, band: lowest–highest). Orange, squares, dashed: the sloppy grader's score; orange, squares,
solid: the true reading accuracy of the same model; blue, round dots: the reading accuracy of the model trained
against the exact checker. The gray line is SFT-240 (mean over 3 seeds).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.6))
for key, col, color, marker, ls, label in (
    ("proxy0.0", 2, ORANGE, "s", "--", "sloppy grader: its own score"),
    ("proxy0.0", 1, ORANGE, "s", "-", "sloppy grader: truly correct"),
    ("b0.0", 1, BLUE, "o", "-", "checker: truly correct"),
):
    curves = 100 * np.array([[c[col] for c in s["curve"]] for s in runs["rl"][key]["seeds"]])
    steps = [c[0] for c in runs["rl"][key]["seeds"][0]["curve"]]
    ax.fill_between(steps, curves.min(0), curves.max(0), color=color, alpha=0.15)
    ax.plot(steps, curves.mean(0), marker=marker, ls=ls, color=color, ms=4, label=label)
ax.axhline(100 * np.mean([s["val"] for s in runs["sft"]["240"]["seeds"]]), color=GREY, lw=1, label="SFT-240")
ax.set(xlabel="reinforcement-learning step (β = 0)", ylabel="validation set (%)", ylim=(30, 100))
ax.set_title("The sloppy grader's score rises, the true reading accuracy doesn't")
ax.legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
ax.grid(axis="y", alpha=0.3)
fmt_axes(fig)
plt.show()
p0 = runs["rl"]["proxy0.0"]["seeds"]
print("KL to the SFT version at step 1,000 (3 seeds):", " · ".join(fmt(s["curve"][-1][4], 2) for s in p0), "nats")

The sloppy grader's score (dashed) rises almost all the way, while the true reading accuracy (solid orange) hovers
around SFT's level and then drops on two seeds, and the KL climbs to over 10 nats on one seed. Three things to
remember:
- **A broken reward doesn't report an error.** Looking only at the grader's score, this run looks like a success.
- **A KL penalty reduces the damage but can't fix a wrong measure.** With β = 0.1, the true accuracy holds at
  62.50–65.58%, still worse than training against the exact checker with the same β (67.58–73.25%).
- **Verifiable rewards** (a program that grades right–wrong, as for math and programming) are much harder to game.
  That is why DeepSeek-R1-Zero learns with only a right–wrong reward and a format reward, without a reward model
  learned from people. But “verifiable” is only safe when the checker really is correct: `proxy` is a program too.

## 11 · Test-time compute: many samples, voting, best-of-N

Instead of training more, we can let the model **think more at answer time**: take $k$ samples for each question,
then
- **majority voting** (self-consistency, Wang et al., 2022): answer with the answer that appears most often, breaking
  ties in favor of the one that appears first; no checker needed;
- **best-of-N with a checker:** answer with a sample the checker accepts, if there is one. With an infallible
  checker, the accuracy is “at least one correct sample out of $k$”, i.e. pass@k (Chen et al., 2021, unbiased
  estimator $1 - \binom{n-c}{k} / \binom{n}{k}$ from $n$ samples of which $c$ are correct).

The cell below takes 64 samples ($T = 1$) for each of the 1,200 test questions, with exactly the sequence of random
numbers the reel uses (`default_rng(9_000)`, drawn in blocks of 8 samples × 4 batches × 13 steps × 2,400 rows), for
SFT-240 and the model trained against the checker.
Many samples share a prefix (“52…”), so the function below computes each state (question, prefix) only once.

In [ ]:
def generate_many(p, X, M, U, rep):
    """Like generate(…, U, rep), but rows with the same question and the same prefix share one computation."""
    Hs, h_T, _ = encode(p, X, M)
    keys = Hs @ p["Ua"]
    n = len(X) * rep
    out, act = np.full((n, TMAX), EOS), np.arange(n)
    node = np.repeat(np.arange(len(X)), rep)  # the current state of each row
    s, y, q = h_T, np.full(len(X), BOS), np.arange(len(X))  # each state: GRU, the token just written, the question
    for i in range(TMAX):
        s, z, _ = logits(p, s, y, Hs[q], keys[q], M[q])
        cum = softmax(z).cumsum(1)[node[act]]
        tok = np.minimum((cum < U[i, act][:, None]).sum(1), N_OUT - 1)
        out[act, i] = tok
        keep = tok != EOS
        act, tok, parent = act[keep], tok[keep], node[act][keep]
        if not len(act):
            break
        uniq, node[act] = np.unique(parent * N_OUT + tok, return_inverse=True)  # the new, distinct prefixes
        s, y, q = s[uniq // N_OUT], uniq % N_OUT, q[uniq // N_OUT]
    return out


K, n_q = 64, len(rows)
u = np.random.default_rng(9_000).random((K // 8, 4, TMAX, 2400))  # 8-sample blocks, 2,400-row batches, step, row
qi, j = np.repeat(np.arange(n_q), K), np.tile(np.arange(K), n_q)
U64 = u[j // 8, (qi * 8 + j % 8) // 2400, :, (qi * 8 + j % 8) % 2400].T  # (13, 1,200 · 64)
print("the reel's random numbers:", " × ".join(fmt(v, 0) for v in u.shape))
TTC = {}
for name in ("sft", "rl"):
    out = generate_many(MODELS[name], X, M, U64, K)
    TTC[name] = {
        "ok": verify(list(np.repeat(rows, K)), out).reshape(n_q, K).astype(bool),
        "px": proxy(list(np.repeat(rows, K)), out).reshape(n_q, K).astype(bool),
        "ans": np.array(text(out)).reshape(n_q, K),
    }
    print(f"{NAME[name]:>16}: {fmt(n_q * K, 0)} samples, {pct(TTC[name]['ok'].mean())} correct")

From those samples we compute two curves as a function of $k$. Voting is measured as the reel does: split each
question's 64 samples into $64/k$ disjoint blocks of $k$ samples, then average over the blocks.

### 🤔 Predict first
With SFT-240 (51.40% correct with one sample), how far does majority voting over 64 samples get: only a few points
higher, about 70%, or close to 90%? And “at least one correct sample out of 64”?

In [ ]:
def pass_at_k(ok, k):
    """The unbiased estimator of Chen et al. (2021): 1 − C(n − c, k) / C(n, k), averaged over the questions."""
    n = ok.shape[1]
    return float(np.mean([1 - math.comb(n - c, k) / math.comb(n, k) for c in ok.sum(1)]))


def majority_at_k(ans, ok, k):
    """Majority voting over disjoint blocks of k samples; a tie goes to the answer that appears first in the block."""
    ids = np.array([np.unique(a, return_inverse=True)[1] for a in ans]).reshape(len(ans), -1, k)
    votes = (ids[:, :, :, None] == ids[:, :, None, :]).sum(-1)  # the number of votes for the answer at each position
    pick = votes.argmax(-1)  # the first position with the most votes
    return float(np.take_along_axis(ok.reshape(len(ok), -1, k), pick[..., None], -1).mean())


KS = [1, 2, 4, 8, 16, 32, 64]
curves = {
    name: {"pass": [pass_at_k(t["ok"], k) for k in KS], "maj": [majority_at_k(t["ans"], t["ok"], k) for k in KS]}
    for name, t in TTC.items()
}
for name, key in (("sft", "sft240"), ("rl", "rl0")):
    c = curves[name]
    print(
        f"{NAME[name]:>16}: voting {pct(c['maj'][0])} → {pct(c['maj'][-1])}"
        f" · with a checker (pass@k) {pct(c['pass'][0])} → {pct(c['pass'][-1])}  (k = 1 → 64)"
    )
    assert np.allclose(c["pass"], runs["ttc"][key]["passk"], atol=2e-3), "differs from the demo"
    assert np.allclose(c["maj"], runs["ttc"][key]["maj"], atol=2e-3), "differs from the demo"

Voting takes SFT-240 from 51.40% to 56.00% and then stalls from about $k = 16$: when the most common wrong answer
outnumbers the correct one (like “520” versus “5200”), more votes only make the wrong answer win more surely. With a
checker, “at least one correct sample” rises to 70.08%: the 14-point gap can only be cashed in when there is a way
to *pick out* the correct sample.

More samples only help on questions the model **sometimes** already gets right. The cell below counts, by number of
digits, the questions (out of 100) with no correct sample at all among the 64, and then compares the two models: how
many of the questions SFT-240 never gets right the reinforcement-learning model does get right, and vice versa.

In [ ]:
c_sft, c_rl = TTC["sft"]["ok"].sum(1), TTC["rl"]["ok"].sum(1)
for name, c in (("sft", c_sft), ("rl", c_rl)):
    print(
        f"{NAME[name]:>16}: no correct sample among 64, d = 1…12:",
        [int((c[digits == d] == 0).sum()) for d in range(1, 13)],
    )
agg = {
    "unsolvedBySft": int((c_sft == 0).sum()),
    "solvedByRl": int(((c_sft == 0) & (c_rl > 0)).sum()),
    "lostByRl": int(((c_sft > 0) & (c_rl == 0)).sum()),
}
print(
    f"SFT-240 never correct: {agg['unsolvedBySft']} questions; reinforcement learning gets {agg['solvedByRl']} of them"
    f" right, and loses {agg['lostByRl']} questions that SFT-240 sometimes gets right"
)
assert agg == {k: runs["agg"][k] for k in agg}, "differs from the demo"
assert [int((c_sft[digits == d] == 0).sum()) for d in range(1, 13)] == runs["ttc"]["sft240"]["byDigit"]["unsolved"]

For 12-digit numbers, 88 of the 100 questions have no correct sample among SFT-240's 64 samples. Reinforcement
learning solves some of them, but it also loses 40 questions that SFT-240 sometimes gets right.

Each sample is graded, so at very large $k$ we don't need to sample: with the exact $\pi(\text{correct})$ of
section 6, the probability of at least one correct sample out of $k$ is $1 - (1 - \pi(\text{correct}))^k$. The figure
below plots both measures for SFT-240 (orange, squares) and the reinforcement-learning model (blue, round dots).
Left: $k = 1 \dots 64$ from the real samples, solid with a checker, dashed for voting. Right: $1 - (1 - \pi)^k$
computed exactly up to $k = 2^{20}$ (log axis).

In [ ]:
k_big = np.logspace(0, 20 * math.log10(2), 61)
exact = {name: [float(np.mean(-np.expm1(k * np.log1p(-pc[name])))) for k in k_big] for name in ("sft", "rl")}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
for name, color, marker in (("sft", ORANGE, "s"), ("rl", BLUE, "o")):
    x = np.log2(KS)
    a1.plot(x, 100 * np.array(curves[name]["pass"]), marker=marker, color=color, label=f"{NAME[name]}: with checker")
    a1.plot(
        x,
        100 * np.array(curves[name]["maj"]),
        marker=marker,
        ls="--",
        color=color,
        mfc="white",
        label=f"{NAME[name]}: voting",
    )
    a2.plot(k_big, 100 * np.array(exact[name]), color=color, marker=marker, markevery=6, label=NAME[name])
a1.set_xticks(np.log2(KS), labels=[str(k) for k in KS])
a1.set(xlabel="number of samples k", ylabel="correct (%)", ylim=(45, 100), title="64 real samples")
a2.set(
    xscale="log",
    xlabel="number of samples k (log scale)",
    ylabel="at least one correct sample (%)",
    ylim=(45, 90),
    title="Computed exactly from π(correct)",
)
a1.legend(frameon=False, fontsize=7.5, loc="upper left", ncol=2)
a2.legend(frameon=False, fontsize=8, loc="lower right")
for ax in (a1, a2):
    ax.grid(alpha=0.3)
fmt_axes(fig)
plt.show()
big = {name: float(np.mean(-np.expm1(2**20 * np.log1p(-pc[name])))) for name in ("sft", "rl")}
cross = next(k for k, s, r in zip(k_big, exact["sft"], exact["rl"]) if s > r)
print(
    f"k = 2²⁰: SFT-240 {pct(big['sft'], 1)} · RL {pct(big['rl'], 1)}; SFT-240 pulls ahead from about k = {fmt(cross, 0)}"
)
for name, key in (("sft", "sft240"), ("rl", "rl0")):
    assert abs(big[name] - runs["ttc"][key]["passExact"][-1]["acc"]) < 1e-3, "differs from the demo"

The two curves cross. Reinforcement learning makes a single sample far more often correct (75.79% versus 51.47%),
better than even SFT-240's 64 samples with a checker. But at huge $k$, SFT-240 comes out slightly ahead:
reinforcement learning piles probability onto the questions it already often gets right, and pushes the probability
of the answer to some other questions down to almost 0. Yue et al. (2025) found exactly this in real reasoning
models: pass@1 rises, while at large $k$ the base model wins on pass@k. Reinforcement learning here mainly
**sharpens** what the model already knows, rather than teaching it anything new.

## 12 · How much KL does best-of-N have?

Best-of-N is also a new policy: take $N$ samples from $\pi_{\text{ref}}$, answer with a sample the checker accepts.
How far does it drift from $\pi_{\text{ref}}$? Many papers use the formula $\log N - (N-1)/N$. Beirami et al. (2024)
proved that it is only an **upper bound**: it is accurate when the $N$ samples are almost always different, and it
can be very loose when the answers often repeat.

With a right–wrong checker and a single correct answer, we can compute the exact KL. Let
$p = \pi_{\text{ref}}(\text{correct})$: best-of-N answers correctly with probability $a = 1 - (1-p)^N$; when no sample
is correct, it returns a wrong answer in exactly the proportions of $\pi_{\text{ref}}$. So

$$D_{\mathrm{KL}}\big(\pi_{\text{BoN}} \,\|\, \pi_{\text{ref}}\big) = a \log \frac{a}{p} + (1 - a) \log \frac{1 - a}{1 - p}
\;\le\; -\log p.$$

In [ ]:
def bon_kl(p, N):
    """Best-of-N with a right–wrong checker: the probability a of being correct and the exact KL to π_ref (nats), for
    each question."""
    a = -np.expm1(N * np.log1p(-p))  # 1 − (1 − p)^N, accurate even when p is very small
    with np.errstate(divide="ignore", invalid="ignore"):
        kl = np.where(a > 0, a * np.log(a / p), 0) + np.where(a < 1, (1 - a) * np.log((1 - a) / (1 - p)), 0)
    return a, kl


def star_binary(p, beta):
    """π*_β with a right–wrong reward: the probability of being correct q = p / (p + (1 − p)e^{−1/β}) and
    KL(π*‖π_ref), for each question."""
    den = p + (1 - p) * math.exp(-1 / beta)
    q = p / den
    return q, np.maximum(-(1 - q) / beta - np.log(den), 0)


def star_at_kl(p, kl):
    """The accuracy of π*_β at the same mean KL: bisection on log β (the KL falls as β grows)."""
    lo, hi = -3.0, 3.0
    for _ in range(60):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if star_binary(p, 10**mid)[1].mean() > kl else (lo, mid)
    return float(star_binary(p, 10**hi)[0].mean())


p_ref = np.clip(pc["sft"], 1e-300, 1 - 1e-16)
for N in KS:
    a, kl = bon_kl(p_ref, N)
    bound = math.log(N) - (N - 1) / N
    stored = next(b for b in runs["ttc"]["sft240"]["bon"] if b["N"] == N)
    print(
        f"N = {N:>2}: correct {pct(a.mean())} · exact KL {fmt(kl.mean(), 3)} nats · formula {fmt(bound, 3)} nats"
        f" · π* at the same KL: correct {pct(star_at_kl(p_ref, kl.mean()))}"
    )
    assert (kl <= bound + 1e-12).all() and (kl <= -np.log(p_ref) + 1e-9).all()  # an upper bound, for each question
    assert abs(a.mean() - stored["acc"]) < 2e-4 and abs(kl.mean() - stored["kl"]) < 2e-4, "differs from the demo"
a64, kl64 = bon_kl(p_ref, 64)
print(f"N = 64: the formula is {fmt((math.log(64) - 63 / 64) / kl64.mean(), 1)} times the true KL")

At $N = 64$, best-of-N is 70.26% correct while drifting only 0.412 nats from $\pi_{\text{ref}}$, whereas the formula
gives 3.175 nats. The formula assumes that the $N$ samples are almost always different. Here they repeat a lot: for
easy questions ($p$ close to 1), best-of-N changes almost nothing, and the same goes for the questions SFT-240 never
gets right ($p \approx 0$). Its KL never exceeds $-\log p$ on any question, however large $N$ is.

The last column compares with $\pi^*_\beta$ at the **same** mean KL (β chosen by bisection): best-of-N is nearly
optimal, only a few tenths of a point behind $\pi^*$. The figure below puts every policy on the same plane: the
accuracy of one sample against the KL to SFT-240 (seed 0). The green line is $\pi^*_\beta$ as β runs from 10 down
to 0.02, the upper limit of every policy at the same KL. Orange, squares: best-of-N, $N = 1 \dots 64$, placed at the
exact KL (filled squares) and at the formula's KL (hollow squares, dashed). Blue, round dots: four real
reinforcement-learning runs with the checker (seed 0, β = 0, 0.03, 0.1, 0.3; read from the results file).

In [ ]:
betas = np.logspace(1, math.log10(0.02), 40)
frontier = np.array([[v.mean() for v in star_binary(p_ref, b)[::-1]] for b in betas])  # (KL, correct)
bon = np.array([[bon_kl(p_ref, N)[1].mean(), bon_kl(p_ref, N)[0].mean()] for N in KS])
rl_pts = [(b, runs["rl"][f"b{b}"]["seeds"][0]["test"]) for b in (0.0, 0.03, 0.1, 0.3)]
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(frontier[:, 0], 100 * frontier[:, 1], color=GREEN, lw=2, label="π*_β (upper limit)")
ax.plot(bon[:, 0], 100 * bon[:, 1], "s-", color=ORANGE, label="best-of-N, exact KL")
ax.plot(
    [math.log(N) - (N - 1) / N for N in KS],
    100 * bon[:, 1],
    "s--",
    color=ORANGE,
    mfc="white",
    label="best-of-N, KL from the formula",
)
ax.plot(
    [t["kl"] for _, t in rl_pts],
    [100 * t["sample"] for _, t in rl_pts],
    "o",
    color=BLUE,
    ms=7,
    label="real reinforcement learning",
)
for b, t in rl_pts:
    ax.annotate(
        f"β = {tick(b)}",
        (t["kl"], 100 * t["sample"]),
        xytext=(6, -12),
        textcoords="offset points",
        fontsize=8,
        color=BLUE,
    )
ax.set(xlabel="KL to SFT-240 (nats)", ylabel="correct with one sample (%)", xlim=(0, 4.5), ylim=(50, 100))
ax.set_title("Trading drift for accuracy: π*, best-of-N and reinforcement learning")
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.grid(alpha=0.3)
fmt_axes(fig)
plt.show()
for b, t in rl_pts:
    print(
        f"RL β = {tick(b)}: correct {pct(t['sample'])} at KL {fmt(t['kl'], 2)} nats · π* at the same KL: "
        f"{pct(star_at_kl(p_ref, t['kl']))}"
    )
assert abs(star_at_kl(p_ref, rl_pts[2][1]["kl"]) - runs["rl"]["b0.1"]["seeds"][0]["starAtKl"]) < 1e-3, (
    "differs from the demo"
)

Three things this figure says:
- **Every policy trades KL for accuracy**, and $\pi^*$ is the upper limit: no policy is more accurate at the same KL.
  Best-of-N stays close to that limit (with an infallible checker) without learning anything, but it costs $N$ times
  the computation for each answer.
- **Real reinforcement learning sits well below the $\pi^*$ line.** $\pi^*$ chooses separately for each question;
  the real model has to share 53,259 parameters across every question, learns for only 1,000 steps with noisy
  gradients, and also has to *generalize* to test questions it has never seen. In exchange, it costs only one sample
  per answer.
- **Don't trust the formula $\log N - (N-1)/N$** when reporting best-of-N's reward–KL curve: here it pushes the
  orange points to the right by up to about 8 times, making best-of-N look worse than reinforcement learning. In
  fact each nat of KL buys it much more accuracy: 70.26% at 0.412 nats, versus 57.64% at 0.99 nats for
  reinforcement learning with β = 0.3.

## 13 · Try changing things

The cells below rerun the same functions with different parameters. You can change the values and run them again.

### Any number
Change `my_number`. The cell prints the three models' greedy answers, the scores of both graders, and 8 samples from
SFT-240.

In [ ]:
my_number = "2050"
Xn, Mn = questions([my_number])
print(f"{my_number}: “{' '.join(read(int(my_number)))}”")
for name, p in MODELS.items():
    out = generate(p, Xn, Mn)
    print(
        f"  {NAME[name]:>20}: {text(out)[0]:>13} · checker {int(verify([my_number], out)[0])}"
        f" · sloppy {int(proxy([my_number], out)[0])} · π(correct) = {prob(math.exp(logprob(p, [my_number], [my_number])[0]))}"
    )
print(
    "  8 samples from SFT-240:",
    text(generate(MODELS["sft"], Xn, Mn, np.random.default_rng(0).random((TMAX, 8)), rep=8)),
)

### π* on another question
`posttrain-cands.json` has 12 questions (printed below). Change `which` and `beta_try` to see how $\pi^*$ piles up
the probability with each reward.

In [ ]:
print("the questions:", " · ".join(c["digits"] for c in cands["questions"]))
which, beta_try = "554000", 0.1
cq = next(c for c in cands["questions"] if c["digits"] == which)
lab_q, pref_q, r_q, px_q = items(cq)
for reward, rv in (("the checker", r_q), ("the sloppy grader", px_q)):
    q, _, kl = star(pref_q, rv, beta_try)
    top3 = " · ".join(f"{lab_q[i]} {pct(q[i])}" for i in np.argsort(-q)[:3])
    print(f"π* with {reward}, β = {fmt(beta_try, 2)}: {top3} · truly correct {pct(q @ r_q)} · KL {fmt(kl, 2)} nats")
print("π_ref:", " · ".join(f"{lab_q[i]} {pct(pref_q[i])}" for i in np.argsort(-pref_q)[:3]))

For 554000, $\pi_{\text{ref}}$ prefers the wrong answer “5540000” (one extra 0) almost 7 times as much as the correct
one, so optimizing the sloppy grader keeps almost only the wrong answer.

### Reinforcement learning against the sloppy grader
20 steps of training against `proxy` with β = 0, from SFT-240 (seed 1). Change `reward`, `beta_try` or `n_steps` and
run again; with a large `n_steps` (a few hundred), the sloppy grader's score rises clearly while the true reading
accuracy doesn't.

In [ ]:
reward, beta_try, n_steps = proxy, 0.0, 20
policy2 = {k: v.copy() for k, v in ref.items()}
opt2, rng2 = Adam(policy2), np.random.default_rng(50_000 + 1)
for _ in range(n_steps):
    rl_step(policy2, ref, opt2, rng2, beta=beta_try, reward=reward)
res = evaluate(policy2, ref, VAL, 1)
print(
    f"after {n_steps} steps: correct {pct(res['correct'])} · sloppy {pct(res['sloppy'])} · KL {fmt(res['KL'], 2)} nats"
    f"  (SFT-240: correct {pct(before['correct'])} · sloppy {pct(before['sloppy'])})"
)

20 steps are too few to see anything clearly; the KL has already risen to a few nats because there is no KL penalty
(β = 0) and η is large enough for each step to change the model quite a lot.

## 14 · Limits, and what this lesson hasn't done

- **Rewards from people.** Here the checker is a program. For a real chatbot, “a good answer” has no exact
  definition; RLHF learns a **reward model** $r(x, y)$ from people's choices, with exactly DPO's loss
  $-\log \sigma(r(y_w) - r(y_l))$ from section 9 (the logistic regression of Lesson 04 on the difference of two
  scores). That reward model is an approximate measure, so section 10 is an everyday matter, not a rare one.
- **Full PPO and GRPO.** We update once per batch of samples. PPO and GRPO reuse each batch for several steps, so they
  need to clip the ratio $\pi_\theta/\pi_{\text{old}}$ so as not to go too far; PPO also learns a value network to get
  an advantage for each token.
- **Long reasoning.** DeepSeek-R1-Zero learns with GRPO using only a right–wrong reward and a `<think>…</think>`
  format reward, and on its own writes longer and longer chains of reasoning; its AIME 2024 score rises from 15.6%
  to 71.0% (first version of the paper, 2025). Our number-reading model answers in one pass, with no room to
  “think”. The full DeepSeek-R1 also starts with SFT on a few thousand example reasoning chains.
- **Does reinforcement learning teach anything new?** Here it mainly sharpens: it raises pass@1 and lowers pass@k at
  very large $k$ (section 11), as Yue et al. (2025) found. Later studies (for example ProRL, Liu et al., 2025) show
  that reinforcement learning run long enough, with a KL penalty and periodic resets of $\pi_{\text{ref}}$, can solve
  even questions the base model never solves. The question is still being debated.

This is the last lesson of the course: from a single neuron and its count of multiply-accumulate operations
(Lesson 01) to a model taught to answer through feedback.

## Summary
- A base model only continues text: GPT-2 small gives “ Paris” a probability of 0.47% after the bare question, and
  67.35% after two question–answer examples. The knowledge is there; SFT teaches the habit of answering.
- Reinforcement learning needs no labels, only a grader: SFT-240 (53.25–56.00%) rises to 75.33–78.17% with the exact
  checker, drifting 3.57–3.99 nats from the SFT version at β = 0; β = 0.1 gives 67.58–73.25% at 1.19–1.96 nats.
  Only 14.9–17.2% of the groups of 8 answers (β = 0) contain both correct and wrong answers; the other groups teach
  nothing.
- The policy gradient: $\nabla J = \mathbb{E}[R\, \nabla \log \pi_\theta(y)]$ with $R = r - \beta \log(\pi_\theta/
  \pi_{\text{ref}})$, the baseline being the mean of the other 7 answers; on a small table it is unbiased and
  converges to $\pi^* \propto \pi_{\text{ref}}\, e^{r/\beta}$.
- DPO learns straight from pairs: with 3,000 pairs, the implicit reward $\beta \log(\pi_\theta/\pi_{\text{ref}})$
  matches $r$ and the policy is close to $\pi^*$.
- Goodhart: trained against the sloppy grader, the grader's score rises to 88.00–92.67% while the true reading
  accuracy is only 44.75–59.67%, below even SFT on 2/3 seeds. A KL penalty reduces the damage but can't fix a wrong
  measure.
- Test-time compute: voting 51.40% → 56.00%, 70.08% with a checker at 64 samples; best-of-64 drifts only 0.412 nats,
  not the 3.175 nats of the formula $\log N - (N-1)/N$. Reinforcement learning wins at 1 sample, SFT is slightly
  ahead at $2^{20}$ samples.

## Review questions
1. Why do two question–answer examples raise the probability of “ Paris” from 0.47% to 67.35% without changing a
   single weight? What does that say about what SFT teaches?
2. Write the gradient of $J = \mathbb{E}_{\pi_\theta}[r] - \beta D_{\mathrm{KL}}(\pi_\theta \| \pi_{\text{ref}})$ in
   REINFORCE form. Why doesn't subtracting a baseline (the mean of the other 7 answers) bias the gradient?
3. In a group of 8 answers that are all wrong, what is the advantage $A_i$ when β = 0? Why does that mean hard
   questions teach nothing?
4. Prove that $\pi^* \propto \pi_{\text{ref}}\, e^{r/\beta}$ is the optimal solution. If $\pi_{\text{ref}}(y) = 0$,
   what is $\pi^*(y)$? How does that relate to the question “does reinforcement learning teach anything new”?
5. For “năm nghìn hai trăm”, why is optimizing the sloppy grader perfectly only 29.46% correct, even though the
   sloppy grader always accepts the correct answer?
6. How many nats does best-of-64 with SFT-240 drift from $\pi_{\text{ref}}$, and how many does the formula
   $\log N - (N-1)/N$ give? Why is the formula so far off for this task?

<details><summary>Answer hints</summary>

1. The two examples only change the *context*: they make the prompt look like a question–answer page, where the next
   line is usually the answer. The knowledge “Paris” is already there from pretraining; SFT teaches behavior (answer,
   then stop), not new knowledge.
2. $\nabla J = \mathbb{E}[R(y)\, \nabla \log \pi_\theta(y)]$, $R = r - \beta \log(\pi_\theta/\pi_{\text{ref}})$.
   The baseline $b$ doesn't depend on $y_i$, so $\mathbb{E}[b\, \nabla \log \pi_\theta(y_i)] = b\, \nabla \sum_y
   \pi_\theta(y) = b\, \nabla 1 = 0$.
3. $r_i = 0$ for every $i$, so $A_i = 0 - 0 = 0$: the gradient is 0. The same goes for an all-correct group. Section 7
   found that only about a quarter of the groups carry a signal.
4. $J(\pi) = -\beta D_{\mathrm{KL}}(\pi \| \pi^*) + \beta \log Z$, largest when $\pi = \pi^*$. $\pi^*(y) = 0$ when
   $\pi_{\text{ref}}(y) = 0$: KL-penalized optimization only redistributes probability among the answers the original
   model could already write.
5. $\pi^*$ for the sloppy grader keeps $\pi_{\text{ref}}$'s ratios among the answers the sloppy grader accepts;
   $\pi_{\text{ref}}$ prefers “520” (70.00%) to “5200” (29.28%), so the correct answer keeps only the share
   29.28 / (70.00 + 29.28 + …) ≈ 29.46%.
6. 0.412 nats versus 3.175 nats. The formula assumes that the $N$ samples are almost always different; here the
   answers repeat a lot, and the true KL on each question never exceeds $-\log \pi_{\text{ref}}(\text{correct})$.
</details>

## Further reading
- L. Ouyang et al. (2022), [Training language models to follow instructions with human feedback](https://arxiv.org/abs/2203.02155)
  (InstructGPT): SFT on about 13 thousand prompts, a reward model, PPO with a per-token KL penalty ($\beta = 0.02$).
- P. Christiano et al. (2017), [Deep RL from Human Preferences](https://arxiv.org/abs/1706.03741); N. Stiennon et
  al. (2020), [Learning to Summarize from Human Feedback](https://arxiv.org/abs/2009.01325).
- C. Zhou et al. (2023), [LIMA: Less Is More for Alignment](https://arxiv.org/abs/2305.11206).
- R. Rafailov et al. (2023), [Direct Preference Optimization](https://arxiv.org/abs/2305.18290).
- L. Gao, J. Schulman, J. Hilton (2022), [Scaling Laws for Reward Model Overoptimization](https://arxiv.org/abs/2210.10760).
- A. Beirami et al. (2024), [Theoretical guarantees on the best-of-n alignment policy](https://arxiv.org/abs/2401.01879):
  $\log N - (N-1)/N$ is only an upper bound on the KL.
- A. Ahmadian et al. (2024), [Back to Basics: Revisiting REINFORCE Style Optimization for RLHF](https://arxiv.org/abs/2402.14740)
  (RLOO, a baseline that is the mean of the other samples).
- Z. Shao et al. (2024), [DeepSeekMath](https://arxiv.org/abs/2402.03300) (GRPO); DeepSeek-AI (2025),
  [DeepSeek-R1](https://arxiv.org/abs/2501.12948); Q. Yu et al. (2025), [DAPO](https://arxiv.org/abs/2503.14476).
- X. Wang et al. (2022), [Self-Consistency](https://arxiv.org/abs/2203.11171); K. Cobbe et al. (2021),
  [Training Verifiers to Solve Math Word Problems](https://arxiv.org/abs/2110.14168).
- C. Snell et al. (2024), [Scaling LLM Test-Time Compute Optimally](https://arxiv.org/abs/2408.03314).
- Y. Yue et al. (2025), [Does RL Really Incentivize Reasoning Capacity Beyond the Base Model?](https://arxiv.org/abs/2504.13837);
  M. Liu et al. (2025), [ProRL](https://arxiv.org/abs/2505.24864), the opposing view.
- N. Lambert, [RLHF Book](https://rlhfbook.com/) (an open book); L. Weng (2024),
  [Reward Hacking in Reinforcement Learning](https://lilianweng.github.io/posts/2024-11-28-reward-hacking/).

**Next lesson:** this is the last lesson of the course. Go back to the
[course overview](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/) to review all 26 lessons, from the Perceptron
to post-training, and pick a lesson you'd like to redo with its notebook.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")